# <span style="color:ORANGE"> GLOBAL DATASET STRUCTURES </span>

## <span style="color:ORANGE">PACKAGES USED</span> ##

In [1]:
from pathlib import Path

import base64
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats

from sklearn.preprocessing import OneHotEncoder

from IPython.display import display



## <span style="color:ORANGE">  GLOBAL CORRELATION </span> ##

In [3]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "03_global_dataset_structure"
)

FEATURE_GROUP = (
    "global_correlation"
)


HIGH_CORRELATION_THRESHOLD = 0.90

TOP_RELATIONSHIPS = 25

MATRIX_CHUNK_SIZE = 100_000

PNG_DPI = 300


# ============================================================
# 02. SOURCE FEATURE DEFINITIONS
# ============================================================

CONTINUOUS_NUMERICAL_FEATURES = [
    "SEND_AGE",
    "TRANS_VALUE"
]


DISCRETE_NUMERICAL_FEATURES = [
    "TRANS_DAY",
    "SEND_POP_REGISTER"
]


CONTINUOUS_GEOGRAPHIC_FEATURES = [
    "SEND_LAT_REGISTER",
    "SEND_LONG_REGISTER",
    "RECEIVE_LAT",
    "RECEIVE_LONG"
]


BINARY_FEATURES = [
    "SEND_GENDER_BE",
    "TRANS_YEAR_BE"
]


FEWF_FEATURES = [
    "TRANS_NUM_CARD_FEWF",
    "SEND_NAME_FEWF",
    "SEND_JOB_FEWF",
    "RECEIVE_LOC_FEWF"
]


OHEWI_FEATURES = [
    "TRANS_WEEK_OHEWI",
    "RECEIVE_CATEGORY_OHEWI"
]


CYCLICAL_FEATURES = [
    "TRANS_MONTH_SIN",
    "TRANS_MONTH_COS",
    "TRANS_HOUR_SIN",
    "TRANS_HOUR_COS"
]


DIRECT_NUMERICAL_FEATURES = (
    CONTINUOUS_NUMERICAL_FEATURES
    +
    DISCRETE_NUMERICAL_FEATURES
    +
    CONTINUOUS_GEOGRAPHIC_FEATURES
    +
    CYCLICAL_FEATURES
)


REQUIRED_SOURCE_FEATURES = (
    DIRECT_NUMERICAL_FEATURES
    +
    BINARY_FEATURES
    +
    FEWF_FEATURES
    +
    OHEWI_FEATURES
)


# ============================================================
# 03. EXCLUDED FEATURES
# ============================================================

EXCLUDED_IDENTIFIER = (
    "NID_ALPHA"
)

EXCLUDED_TARGET = (
    "TARGET_OMEGA"
)


# ============================================================
# 04. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_global_correlation.html"
)


PEARSON_PATH = (
    RESULTS_DIRECTORY
    / "global_pearson_correlation.png"
)


SPEARMAN_PATH = (
    RESULTS_DIRECTORY
    / "global_spearman_correlation.png"
)


ABS_PEARSON_PATH = (
    RESULTS_DIRECTORY
    / "global_absolute_pearson_correlation.png"
)


ABS_SPEARMAN_PATH = (
    RESULTS_DIRECTORY
    / "global_absolute_spearman_correlation.png"
)


PEARSON_SPEARMAN_DIFFERENCE_PATH = (
    RESULTS_DIRECTORY
    / "global_pearson_spearman_difference.png"
)


PAIRWISE_CSV_PATH = (
    RESULTS_DIRECTORY
    / "global_pairwise_correlation.csv"
)


REDUNDANCY_CSV_PATH = (
    RESULTS_DIRECTORY
    / "global_potential_redundancy.csv"
)


FEATURE_MATRIX_OVERVIEW_CSV_PATH = (
    RESULTS_DIRECTORY
    / "global_feature_matrix_overview.csv"
)


# ============================================================
# 06. TEMPORARY WORKING FILES
#
# These files are removed at the end of the analysis.
# ============================================================

TEMPORARY_MATRIX_PATH = (
    RESULTS_DIRECTORY
    / "_temporary_global_feature_matrix.float32.dat"
)


TEMPORARY_RANK_MATRIX_PATH = (
    RESULTS_DIRECTORY
    / "_temporary_global_rank_matrix.float32.dat"
)


for temporary_path in [
    TEMPORARY_MATRIX_PATH,
    TEMPORARY_RANK_MATRIX_PATH
]:

    if temporary_path.exists():

        temporary_path.unlink()


# ============================================================
# 07. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 08. LOAD REQUIRED SOURCE FEATURES
#
# NID_ALPHA and TARGET_OMEGA are deliberately not loaded.
# ============================================================

try:

    dataset_global = pd.read_parquet(
        DATASET_PATH,
        columns=REQUIRED_SOURCE_FEATURES
    )


except Exception as error:

    raise RuntimeError(
        "Unable to load the required global-correlation "
        "features from the parquet dataset."
    ) from error


total_observations = int(
    len(
        dataset_global
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 09. VERIFY SOURCE FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_SOURCE_FEATURES
    if feature not in dataset_global.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 10. SOURCE FEATURE OVERVIEW
# ============================================================

source_overview_records = []


for feature in REQUIRED_SOURCE_FEATURES:

    if feature in CONTINUOUS_NUMERICAL_FEATURES:

        feature_group = (
            "Continuous numerical"
        )


    elif feature in DISCRETE_NUMERICAL_FEATURES:

        feature_group = (
            "Discrete numerical"
        )


    elif feature in CONTINUOUS_GEOGRAPHIC_FEATURES:

        feature_group = (
            "Continuous geographic"
        )


    elif feature in BINARY_FEATURES:

        feature_group = (
            "Binary encoding"
        )


    elif feature in FEWF_FEATURES:

        feature_group = (
            "Frequency encoding with fallback"
        )


    elif feature in OHEWI_FEATURES:

        feature_group = (
            "One-Hot encoding with ignore"
        )


    elif feature in CYCLICAL_FEATURES:

        feature_group = (
            "Cyclical sine/cosine"
        )


    else:

        feature_group = (
            "Undefined"
        )


    source_overview_records.append({

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            feature_group,

        "DATA_TYPE":
            str(
                dataset_global[
                    feature
                ].dtype
            ),

        "MISSING_VALUES":
            int(
                dataset_global[
                    feature
                ]
                .isna()
                .sum()
            ),

        "UNIQUE_VALUES":
            int(
                dataset_global[
                    feature
                ]
                .nunique(
                    dropna=True
                )
            )
    })


source_overview_table = pd.DataFrame(
    source_overview_records
)


# ============================================================
# 11. CREATE FEWF MAPS USING THE EXPLORATORY DATASET
#
# FEWF(category) =
# category count / total dataset observations
#
# Fallback =
# 1 / total dataset observations
#
# This is exploratory only.
#
# A final train/test modeling pipeline should fit these
# mappings on training data only.
# ============================================================

fewf_mappings = {}

fewf_fallback_values = {}

fewf_overview_records = []


for feature in FEWF_FEATURES:

    source_series = (
        dataset_global[
            feature
        ]
    )


    category_counts = (
        source_series
        .value_counts(
            dropna=True
        )
    )


    frequency_map = (
        category_counts
        /
        total_observations
    )


    fallback_value = (
        1.0
        /
        total_observations
    )


    fewf_mappings[
        feature
    ] = (
        frequency_map
    )


    fewf_fallback_values[
        feature
    ] = (
        fallback_value
    )


    fewf_overview_records.append({

        "SOURCE_FEATURE":
            feature,

        "ORIGINAL_CATEGORIES":
            int(
                source_series.nunique(
                    dropna=True
                )
            ),

        "UNIQUE_FREQUENCY_LEVELS":
            int(
                frequency_map.nunique()
            ),

        "MIN_FREQUENCY":
            float(
                frequency_map.min()
            ),

        "MEDIAN_FREQUENCY":
            float(
                frequency_map.median()
            ),

        "MAX_FREQUENCY":
            float(
                frequency_map.max()
            ),

        "FALLBACK_VALUE":
            float(
                fallback_value
            )
    })


fewf_overview_table = pd.DataFrame(
    fewf_overview_records
)


# ============================================================
# 12. COMMON COMPLETE-CASE MASK
#
# A single common sample is used for the entire global
# correlation matrix.
# ============================================================

complete_case_mask = (
    dataset_global[
        REQUIRED_SOURCE_FEATURES
    ]
    .notna()
    .all(
        axis=1
    )
)


missing_case_count = int(
    (
        ~complete_case_mask
    )
    .sum()
)


analysis_source = (
    dataset_global.loc[
        complete_case_mask,
        REQUIRED_SOURCE_FEATURES
    ]
    .copy()
)


# ============================================================
# 13. VALIDATE DIRECT NUMERICAL FEATURES
# ============================================================

for feature in DIRECT_NUMERICAL_FEATURES:

    original_series = (
        analysis_source[
            feature
        ]
    )


    numeric_series = pd.to_numeric(
        original_series,
        errors="coerce"
    )


    invalid_non_numeric_mask = (
        original_series.notna()
        &
        numeric_series.isna()
    )


    invalid_count = int(
        invalid_non_numeric_mask.sum()
    )


    if invalid_count > 0:

        examples = (
            original_series.loc[
                invalid_non_numeric_mask
            ]
            .drop_duplicates()
            .head(
                10
            )
            .tolist()
        )


        raise TypeError(
            f"{feature} contains non-numeric values. "
            f"Invalid observations: {invalid_count}. "
            f"Examples: {examples}"
        )


    analysis_source[
        feature
    ] = (
        numeric_series.astype(
            "float64"
        )
    )


# ============================================================
# 14. VALIDATE BINARY GENDER
#
# Temporary coding:
#
# F = 0
# M = 1
# ============================================================

gender_values = (
    analysis_source[
        "SEND_GENDER_BE"
    ]
    .astype(
        str
    )
)


unexpected_gender_values = sorted(
    set(
        gender_values.unique()
    )
    -
    {
        "F",
        "M"
    }
)


if unexpected_gender_values:

    raise ValueError(
        "Unexpected SEND_GENDER_BE categories: "
        + ", ".join(
            unexpected_gender_values
        )
    )


# ============================================================
# 15. VALIDATE BINARY YEAR
#
# Temporary coding:
#
# 2019 = 0
# 2020 = 1
# ============================================================

year_numeric = pd.to_numeric(
    analysis_source[
        "TRANS_YEAR_BE"
    ],
    errors="coerce"
)


if year_numeric.isna().any():

    raise TypeError(
        "TRANS_YEAR_BE contains non-numeric values."
    )


unexpected_year_values = sorted(
    set(
        year_numeric.unique()
    )
    -
    {
        2019,
        2020
    }
)


if unexpected_year_values:

    raise ValueError(
        "Unexpected TRANS_YEAR_BE values: "
        + ", ".join(
            map(
                str,
                unexpected_year_values
            )
        )
    )


analysis_source[
    "TRANS_YEAR_BE"
] = (
    year_numeric
)


# ============================================================
# 16. REMOVE NON-FINITE NUMERICAL OBSERVATIONS
# ============================================================

finite_numerical_matrix = (
    analysis_source[
        DIRECT_NUMERICAL_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
)


finite_mask = np.isfinite(
    finite_numerical_matrix
).all(
    axis=1
)


non_finite_case_count = int(
    (
        ~finite_mask
    )
    .sum()
)


analysis_source = (
    analysis_source.loc[
        finite_mask
    ]
    .copy()
)


analysis_source.reset_index(
    drop=True,
    inplace=True
)


analysis_observations = int(
    len(
        analysis_source
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite observations are available."
    )


excluded_observations = (
    total_observations
    -
    analysis_observations
)


excluded_percentage = (
    excluded_observations
    /
    total_observations
    *
    100
)


# ============================================================
# 17. CREATE TEMPORARY ONE-HOT REPRESENTATION
#
# handle_unknown="ignore" is retained.
#
# Because fit and transform use the same exploratory sample,
# no unknown category is expected here.
# ============================================================

ohe_source = (
    analysis_source[
        OHEWI_FEATURES
    ]
    .astype(
        str
    )
)


ohe_encoder = OneHotEncoder(

    handle_unknown="ignore",

    sparse_output=True,

    dtype=np.uint8
)


temporary_ohe_matrix = (
    ohe_encoder.fit_transform(
        ohe_source
    )
)


temporary_ohe_matrix = (
    temporary_ohe_matrix.tocsc()
)


ohe_feature_names = (
    ohe_encoder.get_feature_names_out(
        OHEWI_FEATURES
    )
)


ohe_categories = (
    ohe_encoder.categories_
)


ohe_generated_columns = int(
    temporary_ohe_matrix.shape[
        1
    ]
)


ohe_nonzero_entries = int(
    temporary_ohe_matrix.nnz
)


ohe_total_cells = int(
    temporary_ohe_matrix.shape[
        0
    ]
    *
    temporary_ohe_matrix.shape[
        1
    ]
)


ohe_density = (
    ohe_nonzero_entries
    /
    ohe_total_cells
)


ohe_sparsity = (
    1.0
    -
    ohe_density
)


ohe_active_values = np.asarray(
    temporary_ohe_matrix.sum(
        axis=1
    )
).ravel()


ohe_overview_table = pd.DataFrame({

    "METRIC": [
        "Source categorical features",
        "Generated dummy columns",
        "Observations",
        "Non-zero values",
        "Total matrix cells",
        "Density",
        "Sparsity",
        "Minimum active dummies per observation",
        "Mean active dummies per observation",
        "Maximum active dummies per observation"
    ],

    "VALUE": [
        len(
            OHEWI_FEATURES
        ),

        ohe_generated_columns,

        analysis_observations,

        ohe_nonzero_entries,

        ohe_total_cells,

        ohe_density,

        ohe_sparsity,

        float(
            np.min(
                ohe_active_values
            )
        ),

        float(
            np.mean(
                ohe_active_values
            )
        ),

        float(
            np.max(
                ohe_active_values
            )
        )
    ]
})


# ============================================================
# 18. DEFINE FINAL TEMPORARY NUMERICAL MATRIX STRUCTURE
# ============================================================

feature_metadata_records = []


# ------------------------------------------------------------
# Direct numerical features
# ------------------------------------------------------------

for feature in CONTINUOUS_NUMERICAL_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Continuous numerical",

        "REPRESENTATION":
            "Original numerical",

        "STRUCTURAL_SOURCE":
            feature,

        "IS_BINARY":
            False
    })


for feature in DISCRETE_NUMERICAL_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Discrete numerical",

        "REPRESENTATION":
            "Original numerical",

        "STRUCTURAL_SOURCE":
            feature,

        "IS_BINARY":
            False
    })


for feature in CONTINUOUS_GEOGRAPHIC_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Continuous geographic",

        "REPRESENTATION":
            "Original numerical",

        "STRUCTURAL_SOURCE":
            feature,

        "IS_BINARY":
            False
    })


# ------------------------------------------------------------
# Binary features
# ------------------------------------------------------------

for feature in BINARY_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Binary encoding",

        "REPRESENTATION":
            "Temporary binary encoding",

        "STRUCTURAL_SOURCE":
            feature,

        "IS_BINARY":
            True
    })


# ------------------------------------------------------------
# FEWF features
# ------------------------------------------------------------

for feature in FEWF_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Frequency encoding with fallback",

        "REPRESENTATION":
            "Temporary FEWF",

        "STRUCTURAL_SOURCE":
            feature,

        "IS_BINARY":
            False
    })


# ------------------------------------------------------------
# Cyclical features
# ------------------------------------------------------------

for feature in CYCLICAL_FEATURES:

    if feature in [
        "TRANS_MONTH_SIN",
        "TRANS_MONTH_COS"
    ]:

        structural_source = (
            "TRANS_MONTH_CYCLE"
        )


    else:

        structural_source = (
            "TRANS_HOUR_CYCLE"
        )


    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Cyclical sine/cosine",

        "REPRESENTATION":
            "Original cyclical component",

        "STRUCTURAL_SOURCE":
            structural_source,

        "IS_BINARY":
            False
    })


# ------------------------------------------------------------
# One-Hot features
# ------------------------------------------------------------

ohe_name_index = 0


for (
    source_feature,
    categories
) in zip(
    OHEWI_FEATURES,
    ohe_categories
):

    for category in categories:

        generated_name = (
            ohe_feature_names[
                ohe_name_index
            ]
        )


        feature_metadata_records.append({

            "MATRIX_FEATURE":
                generated_name,

            "SOURCE_FEATURE":
                source_feature,

            "FEATURE_GROUP":
                "One-Hot encoding with ignore",

            "REPRESENTATION":
                "Temporary One-Hot dummy",

            "STRUCTURAL_SOURCE":
                source_feature,

            "IS_BINARY":
                True
        })


        ohe_name_index += 1


feature_metadata_table = pd.DataFrame(
    feature_metadata_records
)


matrix_feature_names = (
    feature_metadata_table[
        "MATRIX_FEATURE"
    ]
    .tolist()
)


matrix_feature_count = int(
    len(
        matrix_feature_names
    )
)


# ============================================================
# 19. CREATE DISK-BACKED TEMPORARY FEATURE MATRIX
#
# float32 substantially reduces memory and disk usage.
#
# Correlation accumulation itself is performed in float64.
# ============================================================

global_matrix = np.memmap(

    TEMPORARY_MATRIX_PATH,

    dtype="float32",

    mode="w+",

    shape=(
        analysis_observations,
        matrix_feature_count
    )
)


# ============================================================
# 20. FILL DIRECT NUMERICAL FEATURES
# ============================================================

matrix_column_index = 0


for feature in (
    CONTINUOUS_NUMERICAL_FEATURES
    +
    DISCRETE_NUMERICAL_FEATURES
    +
    CONTINUOUS_GEOGRAPHIC_FEATURES
):

    global_matrix[
        :,
        matrix_column_index
    ] = (
        analysis_source[
            feature
        ]
        .to_numpy(
            dtype="float32"
        )
    )


    matrix_column_index += 1


# ============================================================
# 21. FILL TEMPORARY BINARY FEATURES
# ============================================================

gender_binary = (
    analysis_source[
        "SEND_GENDER_BE"
    ]
    .astype(
        str
    )
    .map({
        "F": 0,
        "M": 1
    })
    .to_numpy(
        dtype="float32"
    )
)


global_matrix[
    :,
    matrix_column_index
] = (
    gender_binary
)


matrix_column_index += 1


year_binary = (
    analysis_source[
        "TRANS_YEAR_BE"
    ]
    .map({
        2019: 0,
        2020: 1
    })
    .to_numpy(
        dtype="float32"
    )
)


global_matrix[
    :,
    matrix_column_index
] = (
    year_binary
)


matrix_column_index += 1


# ============================================================
# 22. FILL TEMPORARY FEWF FEATURES
# ============================================================

fewf_fallback_usage_records = []


for feature in FEWF_FEATURES:

    source_series = (
        analysis_source[
            feature
        ]
    )


    encoded_series = (
        source_series
        .map(
            fewf_mappings[
                feature
            ]
        )
    )


    fallback_mask = (
        encoded_series.isna()
    )


    fallback_uses = int(
        fallback_mask.sum()
    )


    if fallback_uses > 0:

        encoded_series = (
            encoded_series.fillna(
                fewf_fallback_values[
                    feature
                ]
            )
        )


    global_matrix[
        :,
        matrix_column_index
    ] = (
        encoded_series
        .to_numpy(
            dtype="float32"
        )
    )


    fewf_fallback_usage_records.append({

        "FEATURE":
            feature,

        "FALLBACK_VALUE":
            float(
                fewf_fallback_values[
                    feature
                ]
            ),

        "FALLBACK_USES":
            fallback_uses
    })


    matrix_column_index += 1


fewf_fallback_usage_table = pd.DataFrame(
    fewf_fallback_usage_records
)


# ============================================================
# 23. FILL CYCLICAL FEATURES
# ============================================================

for feature in CYCLICAL_FEATURES:

    global_matrix[
        :,
        matrix_column_index
    ] = (
        analysis_source[
            feature
        ]
        .to_numpy(
            dtype="float32"
        )
    )


    matrix_column_index += 1


# ============================================================
# 24. FILL TEMPORARY ONE-HOT DUMMIES
# ============================================================

for dummy_index in range(
    temporary_ohe_matrix.shape[
        1
    ]
):

    dummy_values = (
        temporary_ohe_matrix[
            :,
            dummy_index
        ]
        .toarray()
        .ravel()
        .astype(
            "float32",
            copy=False
        )
    )


    global_matrix[
        :,
        matrix_column_index
    ] = (
        dummy_values
    )


    matrix_column_index += 1


if matrix_column_index != matrix_feature_count:

    raise RuntimeError(
        "Temporary matrix column construction is inconsistent."
    )


global_matrix.flush()


# ============================================================
# 25. RELEASE LARGE SOURCE OBJECTS BEFORE CORRELATION
# ============================================================

del dataset_global
del finite_numerical_matrix

del gender_binary
del year_binary

del ohe_source
del ohe_active_values

del temporary_ohe_matrix

gc.collect()


# ============================================================
# 26. CORRELATION STRENGTH INTERPRETATION
# ============================================================

def interpret_correlation_strength(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 27. MEMORY-EFFICIENT CORRELATION FUNCTION
#
# Computes Pearson correlation from a disk-backed matrix
# without standardizing the entire dataset in memory.
# ============================================================

def calculate_chunked_correlation(
    matrix,
    chunk_size
):

    number_rows = int(
        matrix.shape[
            0
        ]
    )


    number_columns = int(
        matrix.shape[
            1
        ]
    )


    column_sum = np.zeros(
        number_columns,
        dtype="float64"
    )


    cross_product_sum = np.zeros(
        (
            number_columns,
            number_columns
        ),
        dtype="float64"
    )


    for start_row in range(
        0,
        number_rows,
        chunk_size
    ):

        end_row = min(
            start_row
            +
            chunk_size,
            number_rows
        )


        chunk = np.asarray(

            matrix[
                start_row:end_row,
                :
            ],

            dtype="float64"
        )


        column_sum += (
            chunk.sum(
                axis=0
            )
        )


        cross_product_sum += (
            chunk.T
            @
            chunk
        )


        del chunk


    column_mean = (
        column_sum
        /
        number_rows
    )


    centered_cross_product = (

        cross_product_sum

        -

        number_rows
        *
        np.outer(
            column_mean,
            column_mean
        )
    )


    if number_rows > 1:

        column_variance = (
            np.diag(
                centered_cross_product
            )
            /
            (
                number_rows
                -
                1
            )
        )


    else:

        raise ValueError(
            "At least two observations are required."
        )


    column_variance = np.maximum(
        column_variance,
        0
    )


    column_standard_deviation = np.sqrt(
        column_variance
    )


    denominator = (

        (
            number_rows
            -
            1
        )

        *

        np.outer(
            column_standard_deviation,
            column_standard_deviation
        )
    )


    correlation_matrix = np.divide(

        centered_cross_product,

        denominator,

        out=np.full(
            (
                number_columns,
                number_columns
            ),
            np.nan,
            dtype="float64"
        ),

        where=(
            denominator
            >
            0
        )
    )


    for column_index in range(
        number_columns
    ):

        if (
            column_standard_deviation[
                column_index
            ]
            >
            0
        ):

            correlation_matrix[
                column_index,
                column_index
            ] = (
                1.0
            )


    correlation_matrix = np.clip(
        correlation_matrix,
        -1.0,
        1.0
    )


    return (
        correlation_matrix,
        column_mean,
        column_standard_deviation
    )


# ============================================================
# 28. GLOBAL PEARSON CORRELATION
# ============================================================

(
    pearson_matrix,
    global_means,
    global_standard_deviations
) = calculate_chunked_correlation(

    matrix=global_matrix,

    chunk_size=MATRIX_CHUNK_SIZE
)


pearson_table = pd.DataFrame(

    pearson_matrix,

    index=matrix_feature_names,

    columns=matrix_feature_names
)


# ============================================================
# 29. IDENTIFY CONSTANT MATRIX FEATURES
# ============================================================

constant_feature_mask = (
    global_standard_deviations
    ==
    0
)


constant_feature_names = [
    matrix_feature_names[
        index
    ]
    for index in np.where(
        constant_feature_mask
    )[
        0
    ]
]


constant_features_table = pd.DataFrame({

    "CONSTANT_FEATURE":
        constant_feature_names
})


# ============================================================
# 30. CREATE DISK-BACKED RANK MATRIX FOR SPEARMAN
#
# Spearman correlation is Pearson correlation applied
# to ranked values.
#
# Binary features do not need explicit ranking because
# average ranks are an affine transformation of 0/1 values,
# leaving correlation unchanged.
# ============================================================

rank_matrix = np.memmap(

    TEMPORARY_RANK_MATRIX_PATH,

    dtype="float32",

    mode="w+",

    shape=(
        analysis_observations,
        matrix_feature_count
    )
)


binary_matrix_flags = (
    feature_metadata_table[
        "IS_BINARY"
    ]
    .to_numpy(
        dtype=bool
    )
)


for column_index in range(
    matrix_feature_count
):

    feature_name = (
        matrix_feature_names[
            column_index
        ]
    )


    print(
        (
            f"Preparing Spearman ranks "
            f"{column_index + 1}/{matrix_feature_count}: "
            f"{feature_name}"
        )
    )


    if constant_feature_mask[
        column_index
    ]:

        rank_matrix[
            :,
            column_index
        ] = 1.0

        continue


    if binary_matrix_flags[
        column_index
    ]:

        rank_matrix[
            :,
            column_index
        ] = (
            global_matrix[
                :,
                column_index
            ]
        )


    else:

        column_values = np.asarray(

            global_matrix[
                :,
                column_index
            ],

            dtype="float64"
        )


        ranked_values = stats.rankdata(

            column_values,

            method="average"
        )


        rank_matrix[
            :,
            column_index
        ] = (
            ranked_values.astype(
                "float32",
                copy=False
            )
        )


        del column_values
        del ranked_values


    gc.collect()


rank_matrix.flush()


# ============================================================
# 31. GLOBAL SPEARMAN CORRELATION
# ============================================================

(
    spearman_matrix,
    rank_means,
    rank_standard_deviations
) = calculate_chunked_correlation(

    matrix=rank_matrix,

    chunk_size=MATRIX_CHUNK_SIZE
)


spearman_table = pd.DataFrame(

    spearman_matrix,

    index=matrix_feature_names,

    columns=matrix_feature_names
)


# ============================================================
# 32. DERIVED CORRELATION MATRICES
# ============================================================

absolute_pearson_matrix = np.abs(
    pearson_matrix
)


absolute_spearman_matrix = np.abs(
    spearman_matrix
)


pearson_spearman_difference_matrix = (
    pearson_matrix
    -
    spearman_matrix
)


absolute_magnitude_difference_matrix = np.abs(

    absolute_pearson_matrix
    -
    absolute_spearman_matrix
)


absolute_pearson_table = pd.DataFrame(

    absolute_pearson_matrix,

    index=matrix_feature_names,

    columns=matrix_feature_names
)


absolute_spearman_table = pd.DataFrame(

    absolute_spearman_matrix,

    index=matrix_feature_names,

    columns=matrix_feature_names
)


pearson_spearman_difference_table = pd.DataFrame(

    pearson_spearman_difference_matrix,

    index=matrix_feature_names,

    columns=matrix_feature_names
)


# ============================================================
# 33. METADATA LOOKUP
# ============================================================

metadata_lookup = (
    feature_metadata_table
    .set_index(
        "MATRIX_FEATURE"
    )
    .to_dict(
        orient="index"
    )
)


# ============================================================
# 34. PAIR STRUCTURE CLASSIFICATION
# ============================================================

def classify_pair_structure(
    feature_a,
    feature_b
):

    metadata_a = (
        metadata_lookup[
            feature_a
        ]
    )


    metadata_b = (
        metadata_lookup[
            feature_b
        ]
    )


    group_a = (
        metadata_a[
            "FEATURE_GROUP"
        ]
    )


    group_b = (
        metadata_b[
            "FEATURE_GROUP"
        ]
    )


    structural_source_a = (
        metadata_a[
            "STRUCTURAL_SOURCE"
        ]
    )


    structural_source_b = (
        metadata_b[
            "STRUCTURAL_SOURCE"
        ]
    )


    if (
        group_a
        ==
        "One-Hot encoding with ignore"
        and
        group_b
        ==
        "One-Hot encoding with ignore"
        and
        structural_source_a
        ==
        structural_source_b
    ):

        return (
            "Same-source OHE structural exclusivity"
        )


    if (
        group_a
        ==
        "Cyclical sine/cosine"
        and
        group_b
        ==
        "Cyclical sine/cosine"
        and
        structural_source_a
        ==
        structural_source_b
    ):

        return (
            "Same-cycle sine/cosine structural pair"
        )


    return (
        "Ordinary cross-feature relationship"
    )


# ============================================================
# 35. BUILD ALL UNIQUE FEATURE PAIRS
# ============================================================

pairwise_records = []


for first_index in range(
    matrix_feature_count
):

    for second_index in range(
        first_index + 1,
        matrix_feature_count
    ):

        feature_a = (
            matrix_feature_names[
                first_index
            ]
        )


        feature_b = (
            matrix_feature_names[
                second_index
            ]
        )


        pearson_value = float(
            pearson_matrix[
                first_index,
                second_index
            ]
        )


        spearman_value = float(
            spearman_matrix[
                first_index,
                second_index
            ]
        )


        absolute_pearson = abs(
            pearson_value
        )


        absolute_spearman = abs(
            spearman_value
        )


        maximum_absolute_correlation = max(
            absolute_pearson,
            absolute_spearman
        )


        pair_structure = (
            classify_pair_structure(
                feature_a,
                feature_b
            )
        )


        structural_pair = (
            pair_structure
            !=
            "Ordinary cross-feature relationship"
        )


        potential_redundancy = bool(

            maximum_absolute_correlation
            >=
            HIGH_CORRELATION_THRESHOLD

            and

            pair_structure
            !=
            "Same-source OHE structural exclusivity"
        )


        pairwise_records.append({

            "FEATURE_A":
                feature_a,

            "FEATURE_B":
                feature_b,

            "GROUP_A":
                metadata_lookup[
                    feature_a
                ][
                    "FEATURE_GROUP"
                ],

            "GROUP_B":
                metadata_lookup[
                    feature_b
                ][
                    "FEATURE_GROUP"
                ],

            "SOURCE_A":
                metadata_lookup[
                    feature_a
                ][
                    "SOURCE_FEATURE"
                ],

            "SOURCE_B":
                metadata_lookup[
                    feature_b
                ][
                    "SOURCE_FEATURE"
                ],

            "PAIR_STRUCTURE":
                pair_structure,

            "STRUCTURAL_PAIR":
                structural_pair,

            "PEARSON":
                pearson_value,

            "PEARSON_STRENGTH":
                interpret_correlation_strength(
                    pearson_value
                ),

            "SPEARMAN":
                spearman_value,

            "SPEARMAN_STRENGTH":
                interpret_correlation_strength(
                    spearman_value
                ),

            "ABS_PEARSON":
                absolute_pearson,

            "ABS_SPEARMAN":
                absolute_spearman,

            "PEARSON_MINUS_SPEARMAN":
                pearson_value
                -
                spearman_value,

            "ABS_MAGNITUDE_DIFFERENCE":
                abs(
                    absolute_pearson
                    -
                    absolute_spearman
                ),

            "MAX_ABS_CORRELATION":
                maximum_absolute_correlation,

            "POTENTIAL_REDUNDANCY":
                potential_redundancy
        })


pairwise_table = pd.DataFrame(
    pairwise_records
)


# ============================================================
# 36. TOP PEARSON RELATIONSHIPS
# ============================================================

top_pearson_table = (
    pairwise_table
    .sort_values(
        by="ABS_PEARSON",
        ascending=False
    )
    .head(
        TOP_RELATIONSHIPS
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 37. TOP SPEARMAN RELATIONSHIPS
# ============================================================

top_spearman_table = (
    pairwise_table
    .sort_values(
        by="ABS_SPEARMAN",
        ascending=False
    )
    .head(
        TOP_RELATIONSHIPS
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 38. LARGEST PEARSON-SPEARMAN DISAGREEMENTS
#
# This can reveal monotonic but non-linear structure.
# ============================================================

largest_difference_table = (
    pairwise_table
    .sort_values(
        by="ABS_MAGNITUDE_DIFFERENCE",
        ascending=False
    )
    .head(
        TOP_RELATIONSHIPS
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 39. POTENTIAL REDUNDANCY TABLE
# ============================================================

potential_redundancy_table = (
    pairwise_table[
        pairwise_table[
            "POTENTIAL_REDUNDANCY"
        ]
    ]
    .sort_values(
        by="MAX_ABS_CORRELATION",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 40. SAME-SOURCE ONE-HOT STRUCTURAL PAIRS
# ============================================================

same_source_ohe_table = (
    pairwise_table[
        pairwise_table[
            "PAIR_STRUCTURE"
        ]
        ==
        "Same-source OHE structural exclusivity"
    ]
    .sort_values(
        by="MAX_ABS_CORRELATION",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 41. CYCLICAL STRUCTURAL PAIRS
# ============================================================

cyclical_structural_table = (
    pairwise_table[
        pairwise_table[
            "PAIR_STRUCTURE"
        ]
        ==
        "Same-cycle sine/cosine structural pair"
    ]
    .sort_values(
        by="MAX_ABS_CORRELATION",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 42. CORRELATION SUMMARY BY FEATURE-GROUP PAIR
# ============================================================

group_pair_records = []


for _, row in pairwise_table.iterrows():

    group_a = row[
        "GROUP_A"
    ]


    group_b = row[
        "GROUP_B"
    ]


    sorted_groups = sorted(
        [
            group_a,
            group_b
        ]
    )


    group_pair_records.append({

        "GROUP_1":
            sorted_groups[
                0
            ],

        "GROUP_2":
            sorted_groups[
                1
            ],

        "ABS_PEARSON":
            row[
                "ABS_PEARSON"
            ],

        "ABS_SPEARMAN":
            row[
                "ABS_SPEARMAN"
            ]
    })


group_pair_table = pd.DataFrame(
    group_pair_records
)


group_correlation_summary_table = (
    group_pair_table
    .groupby(
        [
            "GROUP_1",
            "GROUP_2"
        ],
        as_index=False
    )
    .agg(

        PAIR_COUNT=(
            "ABS_PEARSON",
            "size"
        ),

        MEAN_ABS_PEARSON=(
            "ABS_PEARSON",
            "mean"
        ),

        MAX_ABS_PEARSON=(
            "ABS_PEARSON",
            "max"
        ),

        MEAN_ABS_SPEARMAN=(
            "ABS_SPEARMAN",
            "mean"
        ),

        MAX_ABS_SPEARMAN=(
            "ABS_SPEARMAN",
            "max"
        )
    )
)


group_correlation_summary_table[
    "MAX_GLOBAL_ASSOCIATION"
] = (
    group_correlation_summary_table[
        [
            "MAX_ABS_PEARSON",
            "MAX_ABS_SPEARMAN"
        ]
    ]
    .max(
        axis=1
    )
)


group_correlation_summary_table = (
    group_correlation_summary_table
    .sort_values(
        by="MAX_GLOBAL_ASSOCIATION",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 43. FEATURE MATRIX OVERVIEW
# ============================================================

feature_matrix_overview_table = (
    feature_metadata_table
    .copy()
)


feature_matrix_overview_table[
    "MEAN"
] = (
    global_means
)


feature_matrix_overview_table[
    "STANDARD_DEVIATION"
] = (
    global_standard_deviations
)


feature_matrix_overview_table[
    "CONSTANT_FEATURE"
] = (
    global_standard_deviations
    ==
    0
)


# ============================================================
# 44. SAVE MACHINE-READABLE TABLES
# ============================================================

pairwise_table.to_csv(
    PAIRWISE_CSV_PATH,
    index=False
)


potential_redundancy_table.to_csv(
    REDUNDANCY_CSV_PATH,
    index=False
)


feature_matrix_overview_table.to_csv(
    FEATURE_MATRIX_OVERVIEW_CSV_PATH,
    index=False
)


# ============================================================
# 45. GLOBAL CORRELATION HEATMAP FUNCTION
# ============================================================

def create_correlation_heatmap(
    matrix,
    labels,
    title,
    output_path,
    absolute=False,
    difference=False
):

    number_features = len(
        labels
    )


    figure_size = max(
        14,
        number_features
        *
        0.48
    )


    fig, ax = plt.subplots(
        figsize=(
            figure_size,
            figure_size
        )
    )


    if difference:

        maximum_absolute_value = float(
            np.nanmax(
                np.abs(
                    matrix
                )
            )
        )


        if (
            not np.isfinite(
                maximum_absolute_value
            )
            or
            maximum_absolute_value
            ==
            0
        ):

            maximum_absolute_value = 1.0


        image = ax.imshow(

            matrix,

            aspect="auto",

            vmin=-maximum_absolute_value,

            vmax=maximum_absolute_value,

            cmap="coolwarm"
        )


    elif absolute:

        image = ax.imshow(

            matrix,

            aspect="auto",

            vmin=0,

            vmax=1
        )


    else:

        image = ax.imshow(

            matrix,

            aspect="auto",

            vmin=-1,

            vmax=1,

            cmap="coolwarm"
        )


    ax.set_xticks(
        np.arange(
            number_features
        )
    )


    ax.set_yticks(
        np.arange(
            number_features
        )
    )


    ax.set_xticklabels(

        labels,

        rotation=90,

        fontsize=7
    )


    ax.set_yticklabels(

        labels,

        fontsize=7
    )


    ax.set_title(
        title
    )


    fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )


    fig.tight_layout()


    fig.savefig(

        output_path,

        format="png",

        dpi=PNG_DPI,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 46. GLOBAL PEARSON HEATMAP
# ============================================================

create_correlation_heatmap(

    matrix=pearson_matrix,

    labels=matrix_feature_names,

    title="Global Pearson correlation",

    output_path=PEARSON_PATH
)


# ============================================================
# 47. GLOBAL SPEARMAN HEATMAP
# ============================================================

create_correlation_heatmap(

    matrix=spearman_matrix,

    labels=matrix_feature_names,

    title="Global Spearman correlation",

    output_path=SPEARMAN_PATH
)


# ============================================================
# 48. ABSOLUTE PEARSON HEATMAP
# ============================================================

create_correlation_heatmap(

    matrix=absolute_pearson_matrix,

    labels=matrix_feature_names,

    title="Global absolute Pearson correlation",

    output_path=ABS_PEARSON_PATH,

    absolute=True
)


# ============================================================
# 49. ABSOLUTE SPEARMAN HEATMAP
# ============================================================

create_correlation_heatmap(

    matrix=absolute_spearman_matrix,

    labels=matrix_feature_names,

    title="Global absolute Spearman correlation",

    output_path=ABS_SPEARMAN_PATH,

    absolute=True
)


# ============================================================
# 50. PEARSON-SPEARMAN DIFFERENCE HEATMAP
# ============================================================

create_correlation_heatmap(

    matrix=pearson_spearman_difference_matrix,

    labels=matrix_feature_names,

    title="Pearson minus Spearman correlation",

    output_path=PEARSON_SPEARMAN_DIFFERENCE_PATH,

    difference=True
)


# ============================================================
# 51. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 52. CONVERT IMAGES TO BASE64
# ============================================================

pearson_base64 = (
    image_to_base64(
        PEARSON_PATH
    )
)


spearman_base64 = (
    image_to_base64(
        SPEARMAN_PATH
    )
)


absolute_pearson_base64 = (
    image_to_base64(
        ABS_PEARSON_PATH
    )
)


absolute_spearman_base64 = (
    image_to_base64(
        ABS_SPEARMAN_PATH
    )
)


difference_base64 = (
    image_to_base64(
        PEARSON_SPEARMAN_DIFFERENCE_PATH
    )
)


# ============================================================
# 53. HTML TABLE FORMATTING FUNCTION
# ============================================================

def format_pairwise_table_for_html(
    dataframe
):

    return dataframe.to_html(

        index=False,

        border=0,

        formatters={

            "PEARSON":
                lambda value:
                    f"{value:.6f}",

            "SPEARMAN":
                lambda value:
                    f"{value:.6f}",

            "ABS_PEARSON":
                lambda value:
                    f"{value:.6f}",

            "ABS_SPEARMAN":
                lambda value:
                    f"{value:.6f}",

            "PEARSON_MINUS_SPEARMAN":
                lambda value:
                    f"{value:.6f}",

            "ABS_MAGNITUDE_DIFFERENCE":
                lambda value:
                    f"{value:.6f}",

            "MAX_ABS_CORRELATION":
                lambda value:
                    f"{value:.6f}"
        }
    )


# ============================================================
# 54. PREPARE HTML TABLES
# ============================================================

source_overview_html = (
    source_overview_table
    .to_html(
        index=False,
        border=0
    )
)


fewf_overview_html = (
    fewf_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MIN_FREQUENCY":
                lambda value:
                    f"{value:.12f}",

            "MEDIAN_FREQUENCY":
                lambda value:
                    f"{value:.12f}",

            "MAX_FREQUENCY":
                lambda value:
                    f"{value:.12f}",

            "FALLBACK_VALUE":
                lambda value:
                    f"{value:.12f}"
        }
    )
)


fewf_fallback_usage_html = (
    fewf_fallback_usage_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "FALLBACK_VALUE":
                lambda value:
                    f"{value:.12f}"
        }
    )
)


ohe_overview_html = (
    ohe_overview_table
    .to_html(
        index=False,
        border=0
    )
)


feature_matrix_overview_html = (
    feature_matrix_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MEAN":
                lambda value:
                    f"{value:.8f}",

            "STANDARD_DEVIATION":
                lambda value:
                    f"{value:.8f}"
        }
    )
)


top_pearson_html = (
    format_pairwise_table_for_html(
        top_pearson_table
    )
)


top_spearman_html = (
    format_pairwise_table_for_html(
        top_spearman_table
    )
)


largest_difference_html = (
    format_pairwise_table_for_html(
        largest_difference_table
    )
)


potential_redundancy_html = (
    format_pairwise_table_for_html(
        potential_redundancy_table
    )
)


same_source_ohe_html = (
    format_pairwise_table_for_html(
        same_source_ohe_table
    )
)


cyclical_structural_html = (
    format_pairwise_table_for_html(
        cyclical_structural_table
    )
)


group_correlation_summary_html = (
    group_correlation_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MEAN_ABS_PEARSON":
                lambda value:
                    f"{value:.6f}",

            "MAX_ABS_PEARSON":
                lambda value:
                    f"{value:.6f}",

            "MEAN_ABS_SPEARMAN":
                lambda value:
                    f"{value:.6f}",

            "MAX_ABS_SPEARMAN":
                lambda value:
                    f"{value:.6f}",

            "MAX_GLOBAL_ASSOCIATION":
                lambda value:
                    f"{value:.6f}"
        }
    )
)


constant_features_html = (
    constant_features_table
    .to_html(
        index=False,
        border=0
    )
)


# ============================================================
# 55. STRONGEST GLOBAL RELATIONSHIPS
# ============================================================

strongest_pearson_row = (
    top_pearson_table.iloc[
        0
    ]
)


strongest_spearman_row = (
    top_spearman_table.iloc[
        0
    ]
)


strongest_pearson_description = (

    f'{strongest_pearson_row["FEATURE_A"]} × '
    f'{strongest_pearson_row["FEATURE_B"]}'
)


strongest_spearman_description = (

    f'{strongest_spearman_row["FEATURE_A"]} × '
    f'{strongest_spearman_row["FEATURE_B"]}'
)


strongest_pearson_value = float(
    strongest_pearson_row[
        "PEARSON"
    ]
)


strongest_spearman_value = float(
    strongest_spearman_row[
        "SPEARMAN"
    ]
)


# ============================================================
# 56. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Global Dataset Correlation Structure
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1600px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 50px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 12px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 7px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 50px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Global Dataset Structure —
Global Correlation
</h1>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

<br>

<strong>Missing-case exclusions:</strong>
{missing_case_count}

<br>

<strong>Non-finite-case exclusions:</strong>
{non_finite_case_count}

<br>

<strong>Source features:</strong>
{len(REQUIRED_SOURCE_FEATURES)}

<br>

<strong>Final temporary numerical matrix features:</strong>
{matrix_feature_count}

</p>


<div class="note">

<strong>Excluded from the global feature matrix:</strong>

<br><br>

{EXCLUDED_IDENTIFIER}

<br>

{EXCLUDED_TARGET}

<br><br>

NID_ALPHA is an identifier and does not represent an
analytical feature.

TARGET_OMEGA is intentionally excluded because the global
correlation analysis describes the explanatory feature
space that may later be used by PCA, t-SNE and GMM.

</div>


<!-- ========================================================
     1. SOURCE STRUCTURE
========================================================= -->


<h2>
1. Source feature structure
</h2>


<div class="table-container">

{source_overview_html}

</div>


<div class="note">

The parquet dataset retains the original categorical
representations for FEWF and OHEWI source features.

All required transformations are generated temporarily
in memory or disk-backed working matrices.

The original parquet dataset is never modified.

</div>


<!-- ========================================================
     2. TEMPORARY ENCODING
========================================================= -->


<h2>
2. Temporary encoding structure
</h2>


<h3>
Frequency Encoding With Fallback
</h3>


<div class="table-container">

{fewf_overview_html}

</div>


<div class="table-container">

{fewf_fallback_usage_html}

</div>


<div class="note">

FEWF is fitted on the complete exploratory dataset only for
EDA.

For a final train/test workflow, frequency mappings should
be fitted on training data and then applied to validation,
test or future observations.

</div>


<h3>
One-Hot Encoding With Ignore
</h3>


<div class="table-container">

{ohe_overview_html}

</div>


<div class="note">

The One-Hot representation is sparse and is generated using
handle_unknown="ignore".

No unknown categories are expected here because the
encoder is fitted and applied to the same exploratory
sample.

In held-out data, unseen categories would produce an
all-zero vector within the corresponding source-feature
dummy group.

</div>


<!-- ========================================================
     3. FINAL GLOBAL MATRIX
========================================================= -->


<h2>
3. Global numerical feature matrix
</h2>


<div class="table-container">

{feature_matrix_overview_html}

</div>


<div class="note">

The matrix shown above represents the numerical structure
that is relevant for later unsupervised modeling.

No standardization is applied before Pearson or Spearman
correlation because linear rescaling does not change
Pearson correlation and does not alter the ranking used by
Spearman correlation.

</div>


<h3>
Constant features
</h3>


<div class="table-container">

{constant_features_html}

</div>


<!-- ========================================================
     4. PEARSON
========================================================= -->


<h2>
4. Global Pearson correlation
</h2>


<div class="chart">

<img
    src="data:image/png;base64,{pearson_base64}"
    alt="Global Pearson correlation"
>

</div>


<div class="chart">

<img
    src="data:image/png;base64,{absolute_pearson_base64}"
    alt="Global absolute Pearson correlation"
>

</div>


<div class="note">

Pearson correlation evaluates linear association.

It is especially important for identifying linear
redundancy that may influence covariance structure and
principal components.

A weak Pearson correlation does not imply statistical
independence or absence of non-linear dependence.

</div>


<h3>
Strongest Pearson relationships
</h3>


<div class="table-container">

{top_pearson_html}

</div>


<p class="result">

Strongest absolute Pearson relationship:

<br>

{strongest_pearson_description}

<br>

Pearson:
{strongest_pearson_value:.6f}

</p>


<!-- ========================================================
     5. SPEARMAN
========================================================= -->


<h2>
5. Global Spearman correlation
</h2>


<div class="chart">

<img
    src="data:image/png;base64,{spearman_base64}"
    alt="Global Spearman correlation"
>

</div>


<div class="chart">

<img
    src="data:image/png;base64,{absolute_spearman_base64}"
    alt="Global absolute Spearman correlation"
>

</div>


<div class="note">

Spearman correlation evaluates monotonic association based
on ranked values.

It is useful for variables with asymmetry, extreme values,
repeated FEWF levels or relationships that are monotonic but
not strongly linear.

</div>


<h3>
Strongest Spearman relationships
</h3>


<div class="table-container">

{top_spearman_html}

</div>


<p class="result">

Strongest absolute Spearman relationship:

<br>

{strongest_spearman_description}

<br>

Spearman:
{strongest_spearman_value:.6f}

</p>


<!-- ========================================================
     6. PEARSON VS SPEARMAN
========================================================= -->


<h2>
6. Pearson versus Spearman
</h2>


<div class="chart">

<img
    src="data:image/png;base64,{difference_base64}"
    alt="Pearson minus Spearman correlation"
>

</div>


<h3>
Largest magnitude disagreements
</h3>


<div class="table-container">

{largest_difference_html}

</div>


<div class="note">

Large differences between absolute Pearson and Spearman
correlations may indicate monotonic relationships that are
not adequately described by a linear association.

This can be particularly relevant for skewed numerical and
frequency-encoded variables.

</div>


<!-- ========================================================
     7. POTENTIAL REDUNDANCY
========================================================= -->


<h2>
7. Potential redundancy
</h2>


<p>

Potential redundancy threshold:

<strong>
maximum(|Pearson|, |Spearman|) >=
{HIGH_CORRELATION_THRESHOLD:.2f}
</strong>

</p>


<div class="table-container">

{potential_redundancy_html}

</div>


<div class="note">

This table identifies relationships that should be reviewed
before modeling.

A high correlation does not automatically result in feature
removal.

The interpretation must consider feature meaning,
encoding structure, PCA behavior, distance geometry and GMM
covariance estimation.

</div>


<!-- ========================================================
     8. ONE-HOT STRUCTURAL CORRELATION
========================================================= -->


<h2>
8. Same-source One-Hot structural relationships
</h2>


<div class="table-container">

{same_source_ohe_html}

</div>


<div class="note">

Dummy variables produced from the same original
categorical feature are mutually exclusive by construction.

For example, an observation cannot simultaneously belong
to two weekdays.

Therefore, negative correlations among same-source dummy
variables are structurally induced by One-Hot Encoding and
should not automatically be interpreted as unexpected
multicollinearity or redundant information.

</div>


<!-- ========================================================
     9. CYCLICAL STRUCTURAL RELATIONSHIPS
========================================================= -->


<h2>
9. Cyclical sine-cosine structural relationships
</h2>


<div class="table-container">

{cyclical_structural_html}

</div>


<div class="note">

TRANS_MONTH_SIN and TRANS_MONTH_COS jointly represent one
cyclical variable.

TRANS_HOUR_SIN and TRANS_HOUR_COS jointly represent another
cyclical variable.

Each pair follows an approximately deterministic circular
constraint:

<br><br>

<strong>
sin² + cos² = 1
</strong>

<br><br>

Pearson and Spearman correlations can be close to zero even
though the components are deterministically related.

Therefore, correlation alone is insufficient to identify
all forms of dependence in the dataset.

</div>


<!-- ========================================================
     10. FEATURE GROUP STRUCTURE
========================================================= -->


<h2>
10. Correlation by feature-group pair
</h2>


<div class="table-container">

{group_correlation_summary_html}

</div>


<div class="note">

This table summarizes the average and strongest absolute
correlations between feature groups.

It provides a higher-level view of how numerical,
geographic, binary, FEWF, One-Hot and cyclical
representations interact inside the final numerical feature
space.

</div>


<!-- ========================================================
     11. PCA IMPLICATIONS
========================================================= -->


<h2>
11. Potential implications for PCA
</h2>


<div class="note">

PCA is constructed from variance and covariance structure.

Strongly correlated explanatory variables can concentrate
variance into a smaller number of principal components.

Highly redundant variables can also cause certain latent
directions to be represented repeatedly in the original
space.

<br><br>

The geographic variables deserve particular attention
because previous within-group EDA identified very strong
sender-receiver coordinate relationships.

<br><br>

Standardization strategy must still be decided before PCA
because the current variables have very different units and
scales.

</div>


<!-- ========================================================
     12. t-SNE IMPLICATIONS
========================================================= -->


<h2>
12. Potential implications for t-SNE
</h2>


<div class="note">

t-SNE depends on neighborhood relationships in the feature
space.

Repeated, highly correlated or strongly scaled features can
change pairwise distances and therefore influence which
observations are considered neighbors.

<br><br>

One-Hot sparsity, FEWF concentration, geographic
redundancy and numerical scale differences should be
reviewed before the final t-SNE representation is
constructed.

</div>


<!-- ========================================================
     13. GMM IMPLICATIONS
========================================================= -->


<h2>
13. Potential implications for GMM
</h2>


<div class="note">

Gaussian Mixture Models estimate component covariance
structures.

Highly correlated or nearly redundant variables can
produce poorly conditioned covariance matrices,
particularly when full covariance is used.

<br><br>

Correlation analysis is therefore an important diagnostic,
but it does not by itself determine whether a feature
should be removed.

The next multicollinearity analysis should evaluate the
conditioning and linear dependence of the complete feature
matrix more directly.

</div>


<!-- ========================================================
     14. SUMMARY
========================================================= -->


<h2>
14. Summary
</h2>


<p class="result">

Original source features:
{len(REQUIRED_SOURCE_FEATURES)}

</p>


<p class="result">

Temporary numerical matrix features:
{matrix_feature_count}

</p>


<p class="result">

Potential high-correlation redundancy pairs:
{len(potential_redundancy_table)}

</p>


<p class="result">

Constant features:
{len(constant_feature_names)}

</p>


<p class="result">

Strongest Pearson relationship:

<br>

{strongest_pearson_description}

<br>

{strongest_pearson_value:.6f}

</p>


<p class="result">

Strongest Spearman relationship:

<br>

{strongest_spearman_description}

<br>

{strongest_spearman_value:.6f}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

A complete temporary numerical representation of the
explanatory dataset was constructed without modifying the
source parquet file.

<br><br>

Original numerical, geographic and cyclical variables were
preserved.

Binary features were temporarily encoded as 0/1.

FEWF representations were generated temporarily from the
original categorical values.

OHEWI features were transformed into sparse temporary
One-Hot dummy variables.

<br><br>

Pearson and Spearman matrices were then calculated for the
complete explanatory feature space.

Strong relationships, possible redundancy, linear versus
monotonic differences, structural One-Hot correlations and
cyclical dependence were documented.

<br><br>

No feature is removed during this analysis.

The next global dataset analysis should evaluate
multicollinearity, numerical conditioning and near-linear
dependence before the final pre-modeling audit.

</div>


</body>

</html>
"""


# ============================================================
# 57. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 58. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GLOBAL DATASET STRUCTURE - GLOBAL CORRELATION"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


print(
    "Original source features:",
    len(
        REQUIRED_SOURCE_FEATURES
    )
)


print(
    "Final temporary numerical features:",
    matrix_feature_count
)


# ============================================================
# 59. DISPLAY FEATURE MATRIX STRUCTURE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GLOBAL FEATURE MATRIX OVERVIEW"
)


print(
    "=" * 100
)


display(
    feature_matrix_overview_table
)


# ============================================================
# 60. DISPLAY STRONGEST PEARSON RELATIONSHIPS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TOP ABSOLUTE PEARSON RELATIONSHIPS"
)


print(
    "=" * 100
)


display(
    top_pearson_table
)


# ============================================================
# 61. DISPLAY STRONGEST SPEARMAN RELATIONSHIPS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TOP ABSOLUTE SPEARMAN RELATIONSHIPS"
)


print(
    "=" * 100
)


display(
    top_spearman_table
)


# ============================================================
# 62. DISPLAY PEARSON-SPEARMAN DIFFERENCES
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "LARGEST PEARSON-SPEARMAN MAGNITUDE DIFFERENCES"
)


print(
    "=" * 100
)


display(
    largest_difference_table
)


# ============================================================
# 63. DISPLAY POTENTIAL REDUNDANCY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "POTENTIAL HIGH-CORRELATION REDUNDANCY"
)


print(
    "=" * 100
)


display(
    potential_redundancy_table
)


# ============================================================
# 64. DISPLAY OHE STRUCTURAL RELATIONSHIPS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SAME-SOURCE ONE-HOT STRUCTURAL RELATIONSHIPS"
)


print(
    "=" * 100
)


display(
    same_source_ohe_table
)


# ============================================================
# 65. DISPLAY CYCLICAL STRUCTURAL RELATIONSHIPS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CYCLICAL STRUCTURAL RELATIONSHIPS"
)


print(
    "=" * 100
)


display(
    cyclical_structural_table
)


# ============================================================
# 66. DISPLAY FEATURE-GROUP SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CORRELATION BY FEATURE-GROUP PAIR"
)


print(
    "=" * 100
)


display(
    group_correlation_summary_table
)


# ============================================================
# 67. DISPLAY MAIN FINDINGS
# ============================================================

print(
    "\nStrongest absolute Pearson relationship:"
)


print(
    strongest_pearson_description
)


print(
    "Pearson:",
    f"{strongest_pearson_value:.6f}"
)


print(
    "\nStrongest absolute Spearman relationship:"
)


print(
    strongest_spearman_description
)


print(
    "Spearman:",
    f"{strongest_spearman_value:.6f}"
)


print(
    "\nPotential redundancy pairs:",
    len(
        potential_redundancy_table
    )
)


print(
    "Constant features:",
    len(
        constant_feature_names
    )
)


# ============================================================
# 68. RELEASE MEMORY BEFORE TEMPORARY FILE REMOVAL
# ============================================================

del pearson_table
del spearman_table

del absolute_pearson_table
del absolute_spearman_table

del pearson_spearman_difference_table

del rank_means
del rank_standard_deviations

del analysis_source

del global_matrix
del rank_matrix

gc.collect()


# ============================================================
# 69. REMOVE TEMPORARY DISK-BACKED MATRICES
# ============================================================

if TEMPORARY_MATRIX_PATH.exists():

    TEMPORARY_MATRIX_PATH.unlink()


if TEMPORARY_RANK_MATRIX_PATH.exists():

    TEMPORARY_RANK_MATRIX_PATH.unlink()


# ============================================================
# 70. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GLOBAL CORRELATION ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nCorrelation images:"
)


print(
    PEARSON_PATH
)


print(
    SPEARMAN_PATH
)


print(
    ABS_PEARSON_PATH
)


print(
    ABS_SPEARMAN_PATH
)


print(
    PEARSON_SPEARMAN_DIFFERENCE_PATH
)


print(
    "\nMachine-readable tables:"
)


print(
    PAIRWISE_CSV_PATH
)


print(
    REDUNDANCY_CSV_PATH
)


print(
    FEATURE_MATRIX_OVERVIEW_CSV_PATH
)


print(
    "\nTemporary working matrices removed:"
)


print(
    not TEMPORARY_MATRIX_PATH.exists()
    and
    not TEMPORARY_RANK_MATRIX_PATH.exists()
)

Preparing Spearman ranks 1/39: SEND_AGE
Preparing Spearman ranks 2/39: TRANS_VALUE
Preparing Spearman ranks 3/39: TRANS_DAY
Preparing Spearman ranks 4/39: SEND_POP_REGISTER
Preparing Spearman ranks 5/39: SEND_LAT_REGISTER
Preparing Spearman ranks 6/39: SEND_LONG_REGISTER
Preparing Spearman ranks 7/39: RECEIVE_LAT
Preparing Spearman ranks 8/39: RECEIVE_LONG
Preparing Spearman ranks 9/39: SEND_GENDER_BE
Preparing Spearman ranks 10/39: TRANS_YEAR_BE
Preparing Spearman ranks 11/39: TRANS_NUM_CARD_FEWF
Preparing Spearman ranks 12/39: SEND_NAME_FEWF
Preparing Spearman ranks 13/39: SEND_JOB_FEWF
Preparing Spearman ranks 14/39: RECEIVE_LOC_FEWF
Preparing Spearman ranks 15/39: TRANS_MONTH_SIN
Preparing Spearman ranks 16/39: TRANS_MONTH_COS
Preparing Spearman ranks 17/39: TRANS_HOUR_SIN
Preparing Spearman ranks 18/39: TRANS_HOUR_COS
Preparing Spearman ranks 19/39: TRANS_WEEK_OHEWI_Friday
Preparing Spearman ranks 20/39: TRANS_WEEK_OHEWI_Monday
Preparing Spearman ranks 21/39: TRANS_WEEK_OHEWI_Satu

,MATRIX_FEATURE,SOURCE_FEATURE,FEATURE_GROUP,REPRESENTATION,STRUCTURAL_SOURCE,IS_BINARY,MEAN,STANDARD_DEVIATION,CONSTANT_FEATURE
0,SEND_AGE,SEND_AGE,Continuous numerical,Original numerical,SEND_AGE,False,52.884141,17.402901,False
1,TRANS_VALUE,TRANS_VALUE,Continuous numerical,Original numerical,TRANS_VALUE,False,70.063567,159.253975,False
2,TRANS_DAY,TRANS_DAY,Discrete numerical,Original numerical,TRANS_DAY,False,15.850756,8.876245,False
3,SEND_POP_REGISTER,SEND_POP_REGISTER,Discrete numerical,Original numerical,SEND_POP_REGISTER,False,88643.674509,301487.618344,False
4,SEND_LAT_REGISTER,SEND_LAT_REGISTER,Continuous geographic,Original numerical,SEND_LAT_REGISTER,False,38.539311,5.071470,False
5,SEND_LONG_REGISTER,SEND_LONG_REGISTER,Continuous geographic,Original numerical,SEND_LONG_REGISTER,False,-90.227832,13.747895,False
6,RECEIVE_LAT,RECEIVE_LAT,Continuous geographic,Original numerical,RECEIVE_LAT,False,38.538976,5.105604,False
7,RECEIVE_LONG,RECEIVE_LONG,Continuous geographic,Original numerical,RECEIVE_LONG,False,-90.227940,13.759692,False
8,SEND_GENDER_BE,SEND_GENDER_BE,Binary encoding,Temporary binary encoding,SEND_GENDER_BE,True,0.452196,0.497710,False
9,TRANS_YEAR_BE,TRANS_YEAR_BE,Binary encoding,Temporary binary encoding,TRANS_YEAR_BE,True,0.500727,0.500000,False



TOP ABSOLUTE PEARSON RELATIONSHIPS


,FEATURE_A,FEATURE_B,GROUP_A,GROUP_B,SOURCE_A,SOURCE_B,PAIR_STRUCTURE,STRUCTURAL_PAIR,PEARSON,PEARSON_STRENGTH,SPEARMAN,SPEARMAN_STRENGTH,ABS_PEARSON,ABS_SPEARMAN,PEARSON_MINUS_SPEARMAN,ABS_MAGNITUDE_DIFFERENCE,MAX_ABS_CORRELATION,POTENTIAL_REDUNDANCY
0,SEND_LONG_REGISTER,RECEIVE_LONG,Continuous geographic,Continuous geographic,SEND_LONG_REGISTER,RECEIVE_LONG,Ordinary cross-feature relationship,False,0.999118,Very strong,0.998413,Very strong,0.999118,0.998413,0.000705,0.000705,0.999118,True
1,SEND_LAT_REGISTER,RECEIVE_LAT,Continuous geographic,Continuous geographic,SEND_LAT_REGISTER,RECEIVE_LAT,Ordinary cross-feature relationship,False,0.993582,Very strong,0.991004,Very strong,0.993582,0.991004,0.002578,0.002578,0.993582,True
2,TRANS_NUM_CARD_FEWF,SEND_NAME_FEWF,Frequency encoding with fallback,Frequency encoding with fallback,TRANS_NUM_CARD_FEWF,SEND_NAME_FEWF,Ordinary cross-feature relationship,False,0.968609,Very strong,0.985841,Very strong,0.968609,0.985841,-0.017232,0.017232,0.985841,True
3,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI_grocery_net,Frequency encoding with fallback,One-Hot encoding with ignore,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI,Ordinary cross-feature relationship,False,-0.410198,Moderate,-0.298338,Weak,0.410198,0.298338,-0.111860,0.111860,0.410198,False
4,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI_travel,Frequency encoding with fallback,One-Hot encoding with ignore,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI,Ordinary cross-feature relationship,False,-0.408404,Moderate,-0.291461,Weak,0.408404,0.291461,-0.116943,0.116943,0.408404,False
5,TRANS_HOUR_SIN,RECEIVE_CATEGORY_OHEWI_gas_transport,Cyclical sine/cosine,One-Hot encoding with ignore,TRANS_HOUR_SIN,RECEIVE_CATEGORY_OHEWI,Ordinary cross-feature relationship,False,0.375384,Moderate,0.354376,Moderate,0.375384,0.354376,0.021008,0.021008,0.375384,False
6,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI_gas_transport,Frequency encoding with fallback,One-Hot encoding with ignore,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI,Ordinary cross-feature relationship,False,0.373055,Moderate,0.502982,Strong,0.373055,0.502982,-0.129927,0.129927,0.502982,False
7,TRANS_NUM_CARD_FEWF,SEND_JOB_FEWF,Frequency encoding with fallback,Frequency encoding with fallback,TRANS_NUM_CARD_FEWF,SEND_JOB_FEWF,Ordinary cross-feature relationship,False,0.358387,Moderate,0.361156,Moderate,0.358387,0.361156,-0.002769,0.002769,0.361156,False
8,SEND_AGE,TRANS_NUM_CARD_FEWF,Continuous numerical,Frequency encoding with fallback,SEND_AGE,TRANS_NUM_CARD_FEWF,Ordinary cross-feature relationship,False,-0.353504,Moderate,-0.361268,Moderate,0.353504,0.361268,0.007764,0.007764,0.361268,False
9,TRANS_HOUR_SIN,RECEIVE_CATEGORY_OHEWI_grocery_pos,Cyclical sine/cosine,One-Hot encoding with ignore,TRANS_HOUR_SIN,RECEIVE_CATEGORY_OHEWI,Ordinary cross-feature relationship,False,0.350670,Moderate,0.330978,Moderate,0.350670,0.330978,0.019692,0.019692,0.350670,False



TOP ABSOLUTE SPEARMAN RELATIONSHIPS


...


CYCLICAL STRUCTURAL RELATIONSHIPS


,FEATURE_A,FEATURE_B,GROUP_A,GROUP_B,SOURCE_A,SOURCE_B,PAIR_STRUCTURE,STRUCTURAL_PAIR,PEARSON,PEARSON_STRENGTH,SPEARMAN,SPEARMAN_STRENGTH,ABS_PEARSON,ABS_SPEARMAN,PEARSON_MINUS_SPEARMAN,ABS_MAGNITUDE_DIFFERENCE,MAX_ABS_CORRELATION,POTENTIAL_REDUNDANCY
0,TRANS_MONTH_SIN,TRANS_MONTH_COS,Cyclical sine/cosine,Cyclical sine/cosine,TRANS_MONTH_SIN,TRANS_MONTH_COS,Same-cycle sine/cosine structural pair,True,-0.089870,Very weak or negligible,-0.078303,Very weak or negligible,0.089870,0.078303,-0.011566,0.011566,0.089870,False
1,TRANS_HOUR_SIN,TRANS_HOUR_COS,Cyclical sine/cosine,Cyclical sine/cosine,TRANS_HOUR_SIN,TRANS_HOUR_COS,Same-cycle sine/cosine structural pair,True,0.000775,Very weak or negligible,0.001712,Very weak or negligible,0.000775,0.001712,-0.000938,0.000938,0.001712,False



CORRELATION BY FEATURE-GROUP PAIR


,GROUP_1,GROUP_2,PAIR_COUNT,MEAN_ABS_PEARSON,MAX_ABS_PEARSON,MEAN_ABS_SPEARMAN,MAX_ABS_SPEARMAN,MAX_GLOBAL_ASSOCIATION
0,Continuous geographic,Continuous geographic,6,0.341882,0.999118,0.401404,0.998413,0.999118
1,Frequency encoding with fallback,Frequency encoding with fallback,6,0.286117,0.968609,0.291502,0.985841,0.985841
2,Frequency encoding with fallback,One-Hot encoding with ignore,84,0.039670,0.410198,0.043272,0.502982,0.502982
3,Cyclical sine/cosine,One-Hot encoding with ignore,84,0.030364,0.375384,0.028785,0.354376,0.375384
4,Continuous numerical,Frequency encoding with fallback,8,0.109772,0.353504,0.147185,0.361268,0.361268
5,Continuous numerical,One-Hot encoding with ignore,42,0.017506,0.094821,0.038557,0.339577,0.339577
6,Continuous geographic,Discrete numerical,8,0.051906,0.154816,0.087743,0.263648,0.263648
7,One-Hot encoding with ignore,One-Hot encoding with ignore,210,0.049437,0.238212,0.049437,0.238212,0.238212
8,Binary encoding,Frequency encoding with fallback,8,0.063677,0.216585,0.062446,0.205532,0.216585
9,Cyclical sine/cosine,Frequency encoding with fallback,16,0.024336,0.118871,0.029384,0.190710,0.190710



Strongest absolute Pearson relationship:
SEND_LONG_REGISTER × RECEIVE_LONG
Pearson: 0.999118

Strongest absolute Spearman relationship:
SEND_LONG_REGISTER × RECEIVE_LONG
Spearman: 0.998413

Potential redundancy pairs: 3
Constant features: 0

GLOBAL CORRELATION ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/global_correlation

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/global_correlation/analysis_global_correlation.html

Correlation images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/global_correlation/global_pearson_correlation.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/global_correlation/global_spearman_correlation.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/global_correlatio

## <span style="color:ORANGE"> MULTICOLLINEARITY</span> ##

In [4]:
from pathlib import Path

import base64
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder

from IPython.display import display


# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "03_global_dataset_structure"
)

FEATURE_GROUP = (
    "multicollinearity"
)


MATRIX_CHUNK_SIZE = 100_000

VIF_ELEVATED_THRESHOLD = 5.0

VIF_HIGH_THRESHOLD = 10.0

CONDITION_INDEX_MODERATE_THRESHOLD = 10.0

CONDITION_INDEX_HIGH_THRESHOLD = 30.0

EIGENVALUE_RELATIVE_TOLERANCE = 1e-10

VIF_NUMERICAL_TOLERANCE = 1e-12

NEAR_NULL_COMPONENTS = 5

TOP_COMPONENT_FEATURES = 6

PNG_DPI = 300


# ============================================================
# 02. SOURCE FEATURE DEFINITIONS
# ============================================================

CONTINUOUS_NUMERICAL_FEATURES = [
    "SEND_AGE",
    "TRANS_VALUE"
]


DISCRETE_NUMERICAL_FEATURES = [
    "TRANS_DAY",
    "SEND_POP_REGISTER"
]


CONTINUOUS_GEOGRAPHIC_FEATURES = [
    "SEND_LAT_REGISTER",
    "SEND_LONG_REGISTER",
    "RECEIVE_LAT",
    "RECEIVE_LONG"
]


BINARY_FEATURES = [
    "SEND_GENDER_BE",
    "TRANS_YEAR_BE"
]


FEWF_FEATURES = [
    "TRANS_NUM_CARD_FEWF",
    "SEND_NAME_FEWF",
    "SEND_JOB_FEWF",
    "RECEIVE_LOC_FEWF"
]


OHEWI_FEATURES = [
    "TRANS_WEEK_OHEWI",
    "RECEIVE_CATEGORY_OHEWI"
]


CYCLICAL_FEATURES = [
    "TRANS_MONTH_SIN",
    "TRANS_MONTH_COS",
    "TRANS_HOUR_SIN",
    "TRANS_HOUR_COS"
]


DIRECT_NUMERICAL_FEATURES = (
    CONTINUOUS_NUMERICAL_FEATURES
    +
    DISCRETE_NUMERICAL_FEATURES
    +
    CONTINUOUS_GEOGRAPHIC_FEATURES
    +
    CYCLICAL_FEATURES
)


REQUIRED_SOURCE_FEATURES = (
    DIRECT_NUMERICAL_FEATURES
    +
    BINARY_FEATURES
    +
    FEWF_FEATURES
    +
    OHEWI_FEATURES
)


# ============================================================
# 03. FEATURES EXCLUDED FROM THE UNSUPERVISED MATRIX
# ============================================================

EXCLUDED_IDENTIFIER = (
    "NID_ALPHA"
)

EXCLUDED_TARGET = (
    "TARGET_OMEGA"
)


# ============================================================
# 04. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_multicollinearity.html"
)


VIF_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_vif.csv"
)


HIGH_VIF_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_high_vif_features.csv"
)


FULL_CONDITION_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_full_condition_diagnostics.csv"
)


REDUCED_CONDITION_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_reduced_condition_diagnostics.csv"
)


OHE_REFERENCE_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_ohe_reference_dummies.csv"
)


FULL_NEAR_NULL_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_full_near_linear_components.csv"
)


REDUCED_NEAR_NULL_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_reduced_near_linear_components.csv"
)


MATRIX_OVERVIEW_CSV_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_feature_matrix_overview.csv"
)


VIF_PLOT_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_vif.png"
)


FULL_EIGENVALUE_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_full_eigenvalue_spectrum.png"
)


REDUCED_EIGENVALUE_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_reduced_eigenvalue_spectrum.png"
)


CONDITION_INDEX_PATH = (
    RESULTS_DIRECTORY
    / "multicollinearity_condition_indices.png"
)


TEMPORARY_MATRIX_PATH = (
    RESULTS_DIRECTORY
    / "_temporary_multicollinearity_matrix.float32.dat"
)


# ============================================================
# 06. REMOVE OLD TEMPORARY MATRIX
# ============================================================

if TEMPORARY_MATRIX_PATH.exists():

    TEMPORARY_MATRIX_PATH.unlink()


# ============================================================
# 07. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 08. LOAD REQUIRED SOURCE FEATURES
#
# NID_ALPHA and TARGET_OMEGA are deliberately excluded.
# ============================================================

dataset_multicollinearity = pd.read_parquet(
    DATASET_PATH,
    columns=REQUIRED_SOURCE_FEATURES
)


total_observations = int(
    len(
        dataset_multicollinearity
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 09. VALIDATE REQUIRED FEATURES
# ============================================================

missing_features = [
    feature
    for feature in REQUIRED_SOURCE_FEATURES
    if feature not in dataset_multicollinearity.columns
]


if missing_features:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 10. SOURCE FEATURE OVERVIEW
# ============================================================

source_overview_records = []


for feature in REQUIRED_SOURCE_FEATURES:

    if feature in CONTINUOUS_NUMERICAL_FEATURES:

        feature_group = (
            "Continuous numerical"
        )


    elif feature in DISCRETE_NUMERICAL_FEATURES:

        feature_group = (
            "Discrete numerical"
        )


    elif feature in CONTINUOUS_GEOGRAPHIC_FEATURES:

        feature_group = (
            "Continuous geographic"
        )


    elif feature in BINARY_FEATURES:

        feature_group = (
            "Binary encoding"
        )


    elif feature in FEWF_FEATURES:

        feature_group = (
            "Frequency encoding with fallback"
        )


    elif feature in OHEWI_FEATURES:

        feature_group = (
            "One-Hot encoding with ignore"
        )


    elif feature in CYCLICAL_FEATURES:

        feature_group = (
            "Cyclical sine/cosine"
        )


    else:

        feature_group = (
            "Undefined"
        )


    source_overview_records.append({

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            feature_group,

        "DATA_TYPE":
            str(
                dataset_multicollinearity[
                    feature
                ].dtype
            ),

        "MISSING_VALUES":
            int(
                dataset_multicollinearity[
                    feature
                ]
                .isna()
                .sum()
            ),

        "UNIQUE_VALUES":
            int(
                dataset_multicollinearity[
                    feature
                ]
                .nunique(
                    dropna=True
                )
            )
    })


source_overview_table = pd.DataFrame(
    source_overview_records
)


# ============================================================
# 11. CREATE TEMPORARY FEWF MAPPINGS
#
# FEWF(category) =
# category count / total dataset observations
#
# Fallback =
# 1 / total dataset observations
#
# Exploratory use only.
# ============================================================

fewf_mappings = {}

fewf_fallback_values = {}

fewf_overview_records = []


for feature in FEWF_FEATURES:

    source_series = (
        dataset_multicollinearity[
            feature
        ]
    )


    category_counts = (
        source_series
        .value_counts(
            dropna=True
        )
    )


    frequency_map = (
        category_counts
        /
        total_observations
    )


    fallback_value = (
        1.0
        /
        total_observations
    )


    fewf_mappings[
        feature
    ] = (
        frequency_map
    )


    fewf_fallback_values[
        feature
    ] = (
        fallback_value
    )


    fewf_overview_records.append({

        "FEATURE":
            feature,

        "ORIGINAL_CATEGORIES":
            int(
                source_series.nunique(
                    dropna=True
                )
            ),

        "UNIQUE_FEWF_LEVELS":
            int(
                frequency_map.nunique()
            ),

        "FALLBACK_VALUE":
            float(
                fallback_value
            )
    })


fewf_overview_table = pd.DataFrame(
    fewf_overview_records
)


# ============================================================
# 12. COMMON COMPLETE-CASE SAMPLE
# ============================================================

complete_case_mask = (
    dataset_multicollinearity[
        REQUIRED_SOURCE_FEATURES
    ]
    .notna()
    .all(
        axis=1
    )
)


missing_case_count = int(
    (
        ~complete_case_mask
    )
    .sum()
)


analysis_source = (
    dataset_multicollinearity.loc[
        complete_case_mask,
        REQUIRED_SOURCE_FEATURES
    ]
    .copy()
)


# ============================================================
# 13. RELEASE ORIGINAL DATAFRAME
# ============================================================

del dataset_multicollinearity

gc.collect()


# ============================================================
# 14. VALIDATE AND CONVERT DIRECT NUMERICAL FEATURES
# ============================================================

finite_mask = np.ones(
    len(
        analysis_source
    ),
    dtype=bool
)


for feature in DIRECT_NUMERICAL_FEATURES:

    source_series = (
        analysis_source[
            feature
        ]
    )


    numeric_series = pd.to_numeric(
        source_series,
        errors="coerce"
    )


    invalid_non_numeric_mask = (
        source_series.notna()
        &
        numeric_series.isna()
    )


    invalid_non_numeric_count = int(
        invalid_non_numeric_mask.sum()
    )


    if invalid_non_numeric_count > 0:

        invalid_examples = (
            source_series.loc[
                invalid_non_numeric_mask
            ]
            .drop_duplicates()
            .head(
                10
            )
            .tolist()
        )


        raise TypeError(
            f"{feature} contains non-numeric values. "
            f"Invalid observations: {invalid_non_numeric_count}. "
            f"Examples: {invalid_examples}"
        )


    numeric_values = (
        numeric_series
        .to_numpy(
            dtype="float64"
        )
    )


    finite_mask &= np.isfinite(
        numeric_values
    )


    analysis_source[
        feature
    ] = (
        numeric_series.astype(
            "float64"
        )
    )


    del numeric_values


# ============================================================
# 15. VALIDATE BINARY FEATURES
# ============================================================

gender_values = (
    analysis_source[
        "SEND_GENDER_BE"
    ]
    .astype(
        str
    )
)


unexpected_gender_values = sorted(
    set(
        gender_values.unique()
    )
    -
    {
        "F",
        "M"
    }
)


if unexpected_gender_values:

    raise ValueError(
        "Unexpected SEND_GENDER_BE categories: "
        + ", ".join(
            unexpected_gender_values
        )
    )


year_numeric = pd.to_numeric(
    analysis_source[
        "TRANS_YEAR_BE"
    ],
    errors="coerce"
)


if year_numeric.isna().any():

    raise TypeError(
        "TRANS_YEAR_BE contains non-numeric values."
    )


unexpected_year_values = sorted(
    set(
        year_numeric.unique()
    )
    -
    {
        2019,
        2020
    }
)


if unexpected_year_values:

    raise ValueError(
        "Unexpected TRANS_YEAR_BE values: "
        + ", ".join(
            map(
                str,
                unexpected_year_values
            )
        )
    )


analysis_source[
    "TRANS_YEAR_BE"
] = (
    year_numeric
)


# ============================================================
# 16. REMOVE NON-FINITE NUMERICAL OBSERVATIONS
# ============================================================

non_finite_case_count = int(
    (
        ~finite_mask
    )
    .sum()
)


analysis_source = (
    analysis_source.loc[
        finite_mask
    ]
    .copy()
)


analysis_source.reset_index(
    drop=True,
    inplace=True
)


analysis_observations = int(
    len(
        analysis_source
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite observations are available."
    )


excluded_observations = (
    total_observations
    -
    analysis_observations
)


excluded_percentage = (
    excluded_observations
    /
    total_observations
    *
    100
)


# ============================================================
# 17. CREATE TEMPORARY ONE-HOT REPRESENTATION
# ============================================================

ohe_source = (
    analysis_source[
        OHEWI_FEATURES
    ]
    .astype(
        str
    )
)


ohe_encoder = OneHotEncoder(

    handle_unknown="ignore",

    sparse_output=True,

    dtype=np.uint8
)


temporary_ohe_matrix = (
    ohe_encoder.fit_transform(
        ohe_source
    )
    .tocsc()
)


ohe_feature_names = (
    ohe_encoder.get_feature_names_out(
        OHEWI_FEATURES
    )
)


ohe_categories = (
    ohe_encoder.categories_
)


# ============================================================
# 18. CREATE FEATURE METADATA
# ============================================================

feature_metadata_records = []


for feature in CONTINUOUS_NUMERICAL_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Continuous numerical",

        "REPRESENTATION":
            "Original numerical",

        "CATEGORY":
            None,

        "IS_OHE":
            False
    })


for feature in DISCRETE_NUMERICAL_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Discrete numerical",

        "REPRESENTATION":
            "Original numerical",

        "CATEGORY":
            None,

        "IS_OHE":
            False
    })


for feature in CONTINUOUS_GEOGRAPHIC_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Continuous geographic",

        "REPRESENTATION":
            "Original numerical",

        "CATEGORY":
            None,

        "IS_OHE":
            False
    })


for feature in BINARY_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Binary encoding",

        "REPRESENTATION":
            "Temporary binary encoding",

        "CATEGORY":
            None,

        "IS_OHE":
            False
    })


for feature in FEWF_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Frequency encoding with fallback",

        "REPRESENTATION":
            "Temporary FEWF",

        "CATEGORY":
            None,

        "IS_OHE":
            False
    })


for feature in CYCLICAL_FEATURES:

    feature_metadata_records.append({

        "MATRIX_FEATURE":
            feature,

        "SOURCE_FEATURE":
            feature,

        "FEATURE_GROUP":
            "Cyclical sine/cosine",

        "REPRESENTATION":
            "Original cyclical component",

        "CATEGORY":
            None,

        "IS_OHE":
            False
    })


ohe_name_index = 0


for (
    source_feature,
    categories
) in zip(
    OHEWI_FEATURES,
    ohe_categories
):

    for category in categories:

        generated_name = (
            ohe_feature_names[
                ohe_name_index
            ]
        )


        feature_metadata_records.append({

            "MATRIX_FEATURE":
                generated_name,

            "SOURCE_FEATURE":
                source_feature,

            "FEATURE_GROUP":
                "One-Hot encoding with ignore",

            "REPRESENTATION":
                "Temporary One-Hot dummy",

            "CATEGORY":
                str(
                    category
                ),

            "IS_OHE":
                True
        })


        ohe_name_index += 1


feature_metadata_table = pd.DataFrame(
    feature_metadata_records
)


matrix_feature_names = (
    feature_metadata_table[
        "MATRIX_FEATURE"
    ]
    .tolist()
)


matrix_feature_count = int(
    len(
        matrix_feature_names
    )
)


# ============================================================
# 19. ESTIMATE TEMPORARY MATRIX SIZE
# ============================================================

estimated_matrix_bytes = (

    analysis_observations
    *
    matrix_feature_count
    *
    np.dtype(
        "float32"
    ).itemsize
)


estimated_matrix_gib = (
    estimated_matrix_bytes
    /
    (
        1024 ** 3
    )
)


print(
    "Estimated temporary matrix size:",
    f"{estimated_matrix_gib:.3f} GiB"
)


# ============================================================
# 20. CREATE DISK-BACKED TEMPORARY MATRIX
# ============================================================

global_matrix = np.memmap(

    TEMPORARY_MATRIX_PATH,

    dtype="float32",

    mode="w+",

    shape=(
        analysis_observations,
        matrix_feature_count
    )
)


# ============================================================
# 21. FILL ORIGINAL NUMERICAL FEATURES
# ============================================================

matrix_column_index = 0


for feature in (
    CONTINUOUS_NUMERICAL_FEATURES
    +
    DISCRETE_NUMERICAL_FEATURES
    +
    CONTINUOUS_GEOGRAPHIC_FEATURES
):

    global_matrix[
        :,
        matrix_column_index
    ] = (
        analysis_source[
            feature
        ]
        .to_numpy(
            dtype="float32"
        )
    )


    matrix_column_index += 1


# ============================================================
# 22. FILL TEMPORARY BINARY FEATURES
#
# SEND_GENDER_BE:
# F = 0
# M = 1
#
# TRANS_YEAR_BE:
# 2019 = 0
# 2020 = 1
# ============================================================

gender_binary = (
    analysis_source[
        "SEND_GENDER_BE"
    ]
    .astype(
        str
    )
    .map({
        "F": 0,
        "M": 1
    })
    .to_numpy(
        dtype="float32"
    )
)


global_matrix[
    :,
    matrix_column_index
] = (
    gender_binary
)


matrix_column_index += 1


year_binary = (
    analysis_source[
        "TRANS_YEAR_BE"
    ]
    .map({
        2019: 0,
        2020: 1
    })
    .to_numpy(
        dtype="float32"
    )
)


global_matrix[
    :,
    matrix_column_index
] = (
    year_binary
)


matrix_column_index += 1


# ============================================================
# 23. FILL TEMPORARY FEWF FEATURES
# ============================================================

fewf_fallback_usage_records = []


for feature in FEWF_FEATURES:

    encoded_series = (
        analysis_source[
            feature
        ]
        .map(
            fewf_mappings[
                feature
            ]
        )
    )


    fallback_mask = (
        encoded_series.isna()
    )


    fallback_uses = int(
        fallback_mask.sum()
    )


    if fallback_uses > 0:

        encoded_series = (
            encoded_series.fillna(
                fewf_fallback_values[
                    feature
                ]
            )
        )


    global_matrix[
        :,
        matrix_column_index
    ] = (
        encoded_series
        .to_numpy(
            dtype="float32"
        )
    )


    fewf_fallback_usage_records.append({

        "FEATURE":
            feature,

        "FALLBACK_VALUE":
            fewf_fallback_values[
                feature
            ],

        "FALLBACK_USES":
            fallback_uses
    })


    matrix_column_index += 1


fewf_fallback_usage_table = pd.DataFrame(
    fewf_fallback_usage_records
)


# ============================================================
# 24. FILL CYCLICAL COMPONENTS
# ============================================================

for feature in CYCLICAL_FEATURES:

    global_matrix[
        :,
        matrix_column_index
    ] = (
        analysis_source[
            feature
        ]
        .to_numpy(
            dtype="float32"
        )
    )


    matrix_column_index += 1


# ============================================================
# 25. FILL TEMPORARY OHE DUMMIES
# ============================================================

for dummy_index in range(
    temporary_ohe_matrix.shape[
        1
    ]
):

    dummy_values = (
        temporary_ohe_matrix[
            :,
            dummy_index
        ]
        .toarray()
        .ravel()
        .astype(
            "float32",
            copy=False
        )
    )


    global_matrix[
        :,
        matrix_column_index
    ] = (
        dummy_values
    )


    matrix_column_index += 1


if matrix_column_index != matrix_feature_count:

    raise RuntimeError(
        "Temporary matrix construction is inconsistent."
    )


global_matrix.flush()


# ============================================================
# 26. RELEASE LARGE OBJECTS
# ============================================================

del ohe_source
del temporary_ohe_matrix

del gender_binary
del year_binary

del analysis_source

gc.collect()


# ============================================================
# 27. CHUNKED CORRELATION FUNCTION
#
# Multicollinearity diagnostics use the correlation matrix,
# which is equivalent to working with standardized features.
#
# This prevents raw unit differences from dominating the
# conditioning diagnostics.
# ============================================================

def calculate_chunked_correlation(
    matrix,
    chunk_size
):

    number_rows = int(
        matrix.shape[
            0
        ]
    )


    number_columns = int(
        matrix.shape[
            1
        ]
    )


    column_sum = np.zeros(
        number_columns,
        dtype="float64"
    )


    cross_product_sum = np.zeros(
        (
            number_columns,
            number_columns
        ),
        dtype="float64"
    )


    for start_row in range(
        0,
        number_rows,
        chunk_size
    ):

        end_row = min(
            start_row
            +
            chunk_size,
            number_rows
        )


        chunk = np.asarray(

            matrix[
                start_row:end_row,
                :
            ],

            dtype="float64"
        )


        column_sum += (
            chunk.sum(
                axis=0
            )
        )


        cross_product_sum += (
            chunk.T
            @
            chunk
        )


        del chunk


    means = (
        column_sum
        /
        number_rows
    )


    centered_cross_product = (

        cross_product_sum

        -

        number_rows
        *
        np.outer(
            means,
            means
        )
    )


    variances = (
        np.diag(
            centered_cross_product
        )

        /

        (
            number_rows
            -
            1
        )
    )


    variances = np.maximum(
        variances,
        0
    )


    standard_deviations = np.sqrt(
        variances
    )


    denominator = (

        (
            number_rows
            -
            1
        )

        *

        np.outer(
            standard_deviations,
            standard_deviations
        )
    )


    correlation_matrix = np.divide(

        centered_cross_product,

        denominator,

        out=np.full(
            (
                number_columns,
                number_columns
            ),
            np.nan,
            dtype="float64"
        ),

        where=(
            denominator
            >
            0
        )
    )


    for column_index in range(
        number_columns
    ):

        if (
            standard_deviations[
                column_index
            ]
            >
            0
        ):

            correlation_matrix[
                column_index,
                column_index
            ] = 1.0


    correlation_matrix = np.clip(
        correlation_matrix,
        -1.0,
        1.0
    )


    return (
        correlation_matrix,
        means,
        standard_deviations
    )


# ============================================================
# 28. CALCULATE FULL GLOBAL CORRELATION MATRIX
#
# This matrix is not plotted again because global
# correlation was already analyzed in the previous stage.
#
# It is required internally for multicollinearity
# diagnostics.
# ============================================================

(
    full_correlation_matrix,
    global_means,
    global_standard_deviations
) = calculate_chunked_correlation(

    matrix=global_matrix,

    chunk_size=MATRIX_CHUNK_SIZE
)


# ============================================================
# 29. IDENTIFY CONSTANT FEATURES
# ============================================================

constant_mask = (
    global_standard_deviations
    ==
    0
)


constant_feature_names = [
    matrix_feature_names[
        index
    ]
    for index in np.where(
        constant_mask
    )[
        0
    ]
]


constant_features_table = pd.DataFrame({

    "CONSTANT_FEATURE":
        constant_feature_names
})


nonconstant_indices = np.where(
    ~constant_mask
)[
    0
]


nonconstant_feature_names = [
    matrix_feature_names[
        index
    ]
    for index in nonconstant_indices
]


full_nonconstant_correlation = (
    full_correlation_matrix[
        np.ix_(
            nonconstant_indices,
            nonconstant_indices
        )
    ]
)


# ============================================================
# 30. MATRIX OVERVIEW
# ============================================================

feature_matrix_overview_table = (
    feature_metadata_table
    .copy()
)


feature_matrix_overview_table[
    "MEAN"
] = (
    global_means
)


feature_matrix_overview_table[
    "STANDARD_DEVIATION"
] = (
    global_standard_deviations
)


feature_matrix_overview_table[
    "CONSTANT_FEATURE"
] = (
    constant_mask
)


# ============================================================
# 31. SELECT ONE TEMPORARY OHE REFERENCE DUMMY PER SOURCE
#
# Full One-Hot sets are structurally linearly dependent
# after centering.
#
# One reference dummy from each OHE source is removed only
# for VIF and reduced conditioning diagnostics.
#
# This does NOT modify the parquet dataset and does NOT
# represent a final modeling decision.
# ============================================================

ohe_reference_records = []

ohe_reference_feature_names = []


for source_feature in OHEWI_FEATURES:

    source_dummy_rows = (
        feature_metadata_table[
            (
                feature_metadata_table[
                    "SOURCE_FEATURE"
                ]
                ==
                source_feature
            )
            &
            (
                feature_metadata_table[
                    "IS_OHE"
                ]
                ==
                True
            )
        ]
    )


    if len(
        source_dummy_rows
    ) > 1:

        reference_row = (
            source_dummy_rows.iloc[
                0
            ]
        )


        reference_feature = (
            reference_row[
                "MATRIX_FEATURE"
            ]
        )


        reference_category = (
            reference_row[
                "CATEGORY"
            ]
        )


        ohe_reference_feature_names.append(
            reference_feature
        )


        ohe_reference_records.append({

            "SOURCE_FEATURE":
                source_feature,

            "REFERENCE_CATEGORY":
                reference_category,

            "TEMPORARILY_EXCLUDED_DUMMY":
                reference_feature,

            "PURPOSE":
                "Remove structural OHE dependence for VIF diagnostics"
        })


ohe_reference_table = pd.DataFrame(
    ohe_reference_records
)


# ============================================================
# 32. BUILD REDUCED DIAGNOSTIC FEATURE SET
# ============================================================

reduced_feature_names = [
    feature
    for feature in nonconstant_feature_names
    if feature not in ohe_reference_feature_names
]


feature_name_to_nonconstant_position = {

    feature:
        position

    for position, feature in enumerate(
        nonconstant_feature_names
    )
}


reduced_positions = [
    feature_name_to_nonconstant_position[
        feature
    ]
    for feature in reduced_feature_names
]


reduced_correlation_matrix = (
    full_nonconstant_correlation[
        np.ix_(
            reduced_positions,
            reduced_positions
        )
    ]
)


# ============================================================
# 33. EIGEN DIAGNOSTIC FUNCTION
# ============================================================

def calculate_eigen_diagnostics(
    correlation_matrix,
    feature_names
):

    symmetric_matrix = (
        correlation_matrix
        +
        correlation_matrix.T
    ) / 2


    eigenvalues, eigenvectors = np.linalg.eigh(
        symmetric_matrix
    )


    eigenvalues = np.asarray(
        eigenvalues,
        dtype="float64"
    )


    eigenvectors = np.asarray(
        eigenvectors,
        dtype="float64"
    )


    largest_eigenvalue = float(
        np.max(
            eigenvalues
        )
    )


    eigen_tolerance = (
        EIGENVALUE_RELATIVE_TOLERANCE
        *
        largest_eigenvalue
    )


    positive_mask = (
        eigenvalues
        >
        eigen_tolerance
    )


    numerical_rank = int(
        np.sum(
            positive_mask
        )
    )


    number_features = int(
        len(
            feature_names
        )
    )


    rank_deficiency = (
        number_features
        -
        numerical_rank
    )


    if rank_deficiency > 0:

        correlation_condition_number = (
            np.inf
        )


        standardized_design_condition_number = (
            np.inf
        )


    else:

        smallest_eigenvalue = float(
            np.min(
                eigenvalues
            )
        )


        correlation_condition_number = (
            largest_eigenvalue
            /
            smallest_eigenvalue
        )


        standardized_design_condition_number = (
            np.sqrt(
                correlation_condition_number
            )
        )


    # --------------------------------------------------------
    # CONDITION INDICES
    # --------------------------------------------------------

    descending_indices = np.argsort(
        eigenvalues
    )[
        ::-1
    ]


    descending_eigenvalues = (
        eigenvalues[
            descending_indices
        ]
    )


    condition_indices = []


    for eigenvalue in descending_eigenvalues:

        if eigenvalue <= eigen_tolerance:

            condition_index = (
                np.inf
            )


        else:

            condition_index = float(
                np.sqrt(
                    largest_eigenvalue
                    /
                    eigenvalue
                )
            )


        condition_indices.append(
            condition_index
        )


    condition_table = pd.DataFrame({

        "DIMENSION":
            np.arange(
                1,
                number_features
                + 1
            ),

        "EIGENVALUE":
            descending_eigenvalues,

        "CONDITION_INDEX":
            condition_indices
    })


    condition_table[
        "DIAGNOSTIC_LEVEL"
    ] = (
        condition_table[
            "CONDITION_INDEX"
        ]
        .apply(
            interpret_condition_index
        )
    )


    # --------------------------------------------------------
    # NEAR-NULL COMPONENTS
    # --------------------------------------------------------

    near_null_records = []


    number_components = min(
        NEAR_NULL_COMPONENTS,
        number_features
    )


    ascending_indices = np.argsort(
        eigenvalues
    )


    for component_rank in range(
        number_components
    ):

        eigen_index = (
            ascending_indices[
                component_rank
            ]
        )


        eigenvalue = float(
            eigenvalues[
                eigen_index
            ]
        )


        eigenvector = (
            eigenvectors[
                :,
                eigen_index
            ]
        )


        absolute_loadings = np.abs(
            eigenvector
        )


        top_indices = np.argsort(
            absolute_loadings
        )[
            ::-1
        ][
            :TOP_COMPONENT_FEATURES
        ]


        top_feature_descriptions = []


        for feature_index in top_indices:

            top_feature_descriptions.append(

                f"{feature_names[feature_index]} "
                f"({eigenvector[feature_index]:.6f})"
            )


        if eigenvalue <= eigen_tolerance:

            condition_index = (
                np.inf
            )


        else:

            condition_index = float(
                np.sqrt(
                    largest_eigenvalue
                    /
                    eigenvalue
                )
            )


        near_null_records.append({

            "SMALLEST_COMPONENT_RANK":
                component_rank
                + 1,

            "EIGENVALUE":
                eigenvalue,

            "CONDITION_INDEX":
                condition_index,

            "TOP_CONTRIBUTING_FEATURES":
                "; ".join(
                    top_feature_descriptions
                )
        })


    near_null_table = pd.DataFrame(
        near_null_records
    )


    summary = {

        "NUMBER_FEATURES":
            number_features,

        "NUMERICAL_RANK":
            numerical_rank,

        "RANK_DEFICIENCY":
            rank_deficiency,

        "LARGEST_EIGENVALUE":
            largest_eigenvalue,

        "SMALLEST_EIGENVALUE":
            float(
                np.min(
                    eigenvalues
                )
            ),

        "EIGEN_TOLERANCE":
            eigen_tolerance,

        "CORRELATION_MATRIX_CONDITION_NUMBER":
            correlation_condition_number,

        "STANDARDIZED_DESIGN_CONDITION_NUMBER":
            standardized_design_condition_number
    }


    return (
        eigenvalues,
        eigenvectors,
        condition_table,
        near_null_table,
        summary
    )


# ============================================================
# 34. CONDITION INDEX INTERPRETATION
# ============================================================

def interpret_condition_index(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    if np.isinf(
        value
    ):

        return (
            "Exact or near-exact linear dependence"
        )


    if value >= CONDITION_INDEX_HIGH_THRESHOLD:

        return (
            "High"
        )


    elif value >= CONDITION_INDEX_MODERATE_THRESHOLD:

        return (
            "Moderate"
        )


    else:

        return (
            "Low"
        )


# ============================================================
# 35. FULL MATRIX EIGEN DIAGNOSTICS
# ============================================================

(
    full_eigenvalues,
    full_eigenvectors,
    full_condition_table,
    full_near_null_table,
    full_diagnostic_summary
) = calculate_eigen_diagnostics(

    correlation_matrix=full_nonconstant_correlation,

    feature_names=nonconstant_feature_names
)


# ============================================================
# 36. REDUCED MATRIX EIGEN DIAGNOSTICS
# ============================================================

(
    reduced_eigenvalues,
    reduced_eigenvectors,
    reduced_condition_table,
    reduced_near_null_table,
    reduced_diagnostic_summary
) = calculate_eigen_diagnostics(

    correlation_matrix=reduced_correlation_matrix,

    feature_names=reduced_feature_names
)


# ============================================================
# 37. DIAGNOSTIC SUMMARY TABLE
# ============================================================

diagnostic_summary_table = pd.DataFrame({

    "MATRIX": [
        "Full encoded matrix",
        "Reduced VIF diagnostic matrix"
    ],

    "NUMBER_FEATURES": [
        full_diagnostic_summary[
            "NUMBER_FEATURES"
        ],

        reduced_diagnostic_summary[
            "NUMBER_FEATURES"
        ]
    ],

    "NUMERICAL_RANK": [
        full_diagnostic_summary[
            "NUMERICAL_RANK"
        ],

        reduced_diagnostic_summary[
            "NUMERICAL_RANK"
        ]
    ],

    "RANK_DEFICIENCY": [
        full_diagnostic_summary[
            "RANK_DEFICIENCY"
        ],

        reduced_diagnostic_summary[
            "RANK_DEFICIENCY"
        ]
    ],

    "LARGEST_EIGENVALUE": [
        full_diagnostic_summary[
            "LARGEST_EIGENVALUE"
        ],

        reduced_diagnostic_summary[
            "LARGEST_EIGENVALUE"
        ]
    ],

    "SMALLEST_EIGENVALUE": [
        full_diagnostic_summary[
            "SMALLEST_EIGENVALUE"
        ],

        reduced_diagnostic_summary[
            "SMALLEST_EIGENVALUE"
        ]
    ],

    "CORRELATION_MATRIX_CONDITION_NUMBER": [
        full_diagnostic_summary[
            "CORRELATION_MATRIX_CONDITION_NUMBER"
        ],

        reduced_diagnostic_summary[
            "CORRELATION_MATRIX_CONDITION_NUMBER"
        ]
    ],

    "STANDARDIZED_DESIGN_CONDITION_NUMBER": [
        full_diagnostic_summary[
            "STANDARDIZED_DESIGN_CONDITION_NUMBER"
        ],

        reduced_diagnostic_summary[
            "STANDARDIZED_DESIGN_CONDITION_NUMBER"
        ]
    ]
})


# ============================================================
# 38. VIF INTERPRETATION
# ============================================================

def interpret_vif(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    if np.isinf(
        value
    ):

        return (
            "Infinite / exact dependence"
        )


    if value >= VIF_HIGH_THRESHOLD:

        return (
            "High"
        )


    elif value >= VIF_ELEVATED_THRESHOLD:

        return (
            "Elevated"
        )


    else:

        return (
            "Low"
        )


# ============================================================
# 39. VIF CALCULATION FROM CORRELATION MATRIX
#
# For standardized variables:
#
# VIF = 1 / (1 - R²)
#
# R² is obtained from the correlation matrix.
#
# A pseudoinverse is used for the predictor submatrix so
# that remaining near-singular structure can still be
# diagnosed rather than causing the notebook to fail.
# ============================================================

def calculate_vif_table(
    correlation_matrix,
    feature_names
):

    number_features = int(
        len(
            feature_names
        )
    )


    vif_records = []


    for feature_index in range(
        number_features
    ):

        other_indices = [
            index
            for index in range(
                number_features
            )
            if index != feature_index
        ]


        if len(
            other_indices
        ) == 0:

            multiple_r_squared = (
                0.0
            )


        else:

            predictor_correlation = (
                correlation_matrix[
                    np.ix_(
                        other_indices,
                        other_indices
                    )
                ]
            )


            response_correlations = (
                correlation_matrix[
                    other_indices,
                    feature_index
                ]
            )


            predictor_inverse = np.linalg.pinv(

                predictor_correlation,

                rcond=VIF_NUMERICAL_TOLERANCE
            )


            multiple_r_squared = float(

                response_correlations.T

                @
                predictor_inverse

                @
                response_correlations
            )


            multiple_r_squared = float(
                np.clip(
                    multiple_r_squared,
                    0.0,
                    1.0
                )
            )


        tolerance = (
            1.0
            -
            multiple_r_squared
        )


        if tolerance <= VIF_NUMERICAL_TOLERANCE:

            vif = (
                np.inf
            )


        else:

            vif = float(
                1.0
                /
                tolerance
            )


        vif_records.append({

            "FEATURE":
                feature_names[
                    feature_index
                ],

            "MULTIPLE_R_SQUARED":
                multiple_r_squared,

            "TOLERANCE":
                tolerance,

            "VIF":
                vif,

            "VIF_LEVEL":
                interpret_vif(
                    vif
                )
        })


    vif_table = pd.DataFrame(
        vif_records
    )


    vif_table[
        "_VIF_SORT"
    ] = (
        vif_table[
            "VIF"
        ]
        .replace(
            np.inf,
            np.finfo(
                "float64"
            ).max
        )
    )


    vif_table = (
        vif_table
        .sort_values(
            by="_VIF_SORT",
            ascending=False
        )
        .drop(
            columns=[
                "_VIF_SORT"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return (
        vif_table
    )


# ============================================================
# 40. CALCULATE VIF ON REDUCED DIAGNOSTIC MATRIX
# ============================================================

vif_table = calculate_vif_table(

    correlation_matrix=reduced_correlation_matrix,

    feature_names=reduced_feature_names
)


high_vif_table = (
    vif_table[
        (
            vif_table[
                "VIF"
            ]
            >=
            VIF_HIGH_THRESHOLD
        )
        |
        (
            np.isinf(
                vif_table[
                    "VIF"
                ]
            )
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


elevated_or_high_vif_table = (
    vif_table[
        (
            vif_table[
                "VIF"
            ]
            >=
            VIF_ELEVATED_THRESHOLD
        )
        |
        (
            np.isinf(
                vif_table[
                    "VIF"
                ]
            )
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 41. ADD METADATA TO VIF TABLE
# ============================================================

metadata_lookup = (
    feature_metadata_table
    .set_index(
        "MATRIX_FEATURE"
    )
    .to_dict(
        orient="index"
    )
)


vif_table[
    "SOURCE_FEATURE"
] = (
    vif_table[
        "FEATURE"
    ]
    .map(
        lambda feature:
            metadata_lookup[
                feature
            ][
                "SOURCE_FEATURE"
            ]
    )
)


vif_table[
    "FEATURE_GROUP"
] = (
    vif_table[
        "FEATURE"
    ]
    .map(
        lambda feature:
            metadata_lookup[
                feature
            ][
                "FEATURE_GROUP"
            ]
    )
)


vif_table = (
    vif_table[
        [
            "FEATURE",
            "SOURCE_FEATURE",
            "FEATURE_GROUP",
            "MULTIPLE_R_SQUARED",
            "TOLERANCE",
            "VIF",
            "VIF_LEVEL"
        ]
    ]
)


high_vif_table = (
    vif_table[
        (
            vif_table[
                "VIF"
            ]
            >=
            VIF_HIGH_THRESHOLD
        )
        |
        (
            np.isinf(
                vif_table[
                    "VIF"
                ]
            )
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 42. SAVE MACHINE-READABLE TABLES
# ============================================================

vif_table.to_csv(
    VIF_CSV_PATH,
    index=False
)


high_vif_table.to_csv(
    HIGH_VIF_CSV_PATH,
    index=False
)


full_condition_table.to_csv(
    FULL_CONDITION_CSV_PATH,
    index=False
)


reduced_condition_table.to_csv(
    REDUCED_CONDITION_CSV_PATH,
    index=False
)


ohe_reference_table.to_csv(
    OHE_REFERENCE_CSV_PATH,
    index=False
)


full_near_null_table.to_csv(
    FULL_NEAR_NULL_CSV_PATH,
    index=False
)


reduced_near_null_table.to_csv(
    REDUCED_NEAR_NULL_CSV_PATH,
    index=False
)


feature_matrix_overview_table.to_csv(
    MATRIX_OVERVIEW_CSV_PATH,
    index=False
)


# ============================================================
# 43. CREATE VIF PLOT
# ============================================================

vif_plot_table = (
    vif_table
    .sort_values(
        by="VIF",
        ascending=True
    )
    .copy()
)


finite_vif_values = (
    vif_plot_table.loc[
        np.isfinite(
            vif_plot_table[
                "VIF"
            ]
        ),
        "VIF"
    ]
)


if len(
    finite_vif_values
) > 0:

    finite_vif_max = float(
        finite_vif_values.max()
    )


else:

    finite_vif_max = (
        VIF_HIGH_THRESHOLD
    )


infinite_plot_value = max(

    finite_vif_max
    *
    1.10,

    VIF_HIGH_THRESHOLD
    *
    1.10
)


vif_plot_table[
    "_PLOT_VIF"
] = (
    vif_plot_table[
        "VIF"
    ]
    .replace(
        np.inf,
        infinite_plot_value
    )
)


fig_height = max(
    8,
    len(
        vif_plot_table
    )
    *
    0.32
)


fig, ax = plt.subplots(
    figsize=(
        12,
        fig_height
    )
)


bars = ax.barh(

    vif_plot_table[
        "FEATURE"
    ],

    vif_plot_table[
        "_PLOT_VIF"
    ]
)


ax.axvline(

    VIF_ELEVATED_THRESHOLD,

    linestyle="--",

    linewidth=1,

    label=(
        f"VIF = {VIF_ELEVATED_THRESHOLD:.0f}"
    )
)


ax.axvline(

    VIF_HIGH_THRESHOLD,

    linestyle="--",

    linewidth=1,

    label=(
        f"VIF = {VIF_HIGH_THRESHOLD:.0f}"
    )
)


for bar, vif_value in zip(

    bars,

    vif_plot_table[
        "VIF"
    ]
):

    if np.isinf(
        vif_value
    ):

        ax.text(

            bar.get_width(),

            bar.get_y()
            +
            bar.get_height()
            /
            2,

            " ∞",

            va="center"
        )


ax.set_xlabel(
    "Variance Inflation Factor"
)


ax.set_ylabel(
    "Feature"
)


ax.set_title(
    "VIF after temporary removal of one reference dummy per OHE source"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.20
)


fig.tight_layout()


fig.savefig(

    VIF_PLOT_PATH,

    format="png",

    dpi=PNG_DPI,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 44. EIGENVALUE SPECTRUM PLOT FUNCTION
# ============================================================

def create_eigenvalue_plot(
    eigenvalues,
    title,
    output_path
):

    sorted_eigenvalues = np.sort(
        eigenvalues
    )[
        ::-1
    ]


    largest_value = float(
        np.max(
            sorted_eigenvalues
        )
    )


    positive_floor = max(

        largest_value
        *
        1e-14,

        np.finfo(
            "float64"
        ).tiny
    )


    plot_values = np.maximum(
        sorted_eigenvalues,
        positive_floor
    )


    fig, ax = plt.subplots(
        figsize=(
            11,
            7
        )
    )


    ax.plot(

        np.arange(
            1,
            len(
                plot_values
            )
            +
            1
        ),

        plot_values,

        marker="o"
    )


    ax.set_yscale(
        "log"
    )


    ax.set_xlabel(
        "Eigenvalue rank"
    )


    ax.set_ylabel(
        "Eigenvalue - logarithmic scale"
    )


    ax.set_title(
        title
    )


    ax.grid(
        alpha=0.20
    )


    fig.tight_layout()


    fig.savefig(

        output_path,

        format="png",

        dpi=PNG_DPI,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 45. FULL MATRIX EIGENVALUE PLOT
# ============================================================

create_eigenvalue_plot(

    eigenvalues=full_eigenvalues,

    title=(
        "Eigenvalue spectrum - full encoded correlation matrix"
    ),

    output_path=FULL_EIGENVALUE_PATH
)


# ============================================================
# 46. REDUCED MATRIX EIGENVALUE PLOT
# ============================================================

create_eigenvalue_plot(

    eigenvalues=reduced_eigenvalues,

    title=(
        "Eigenvalue spectrum - reduced VIF diagnostic matrix"
    ),

    output_path=REDUCED_EIGENVALUE_PATH
)


# ============================================================
# 47. CONDITION INDEX PLOT
# ============================================================

condition_plot_table = (
    reduced_condition_table
    .copy()
)


finite_condition_indices = (
    condition_plot_table.loc[
        np.isfinite(
            condition_plot_table[
                "CONDITION_INDEX"
            ]
        ),
        "CONDITION_INDEX"
    ]
)


if len(
    finite_condition_indices
) > 0:

    finite_condition_max = float(
        finite_condition_indices.max()
    )


else:

    finite_condition_max = (
        CONDITION_INDEX_HIGH_THRESHOLD
    )


infinite_condition_plot_value = max(

    finite_condition_max
    *
    1.10,

    CONDITION_INDEX_HIGH_THRESHOLD
    *
    1.10
)


condition_plot_table[
    "_PLOT_CONDITION_INDEX"
] = (
    condition_plot_table[
        "CONDITION_INDEX"
    ]
    .replace(
        np.inf,
        infinite_condition_plot_value
    )
)


fig, ax = plt.subplots(
    figsize=(
        12,
        7
    )
)


ax.plot(

    condition_plot_table[
        "DIMENSION"
    ],

    condition_plot_table[
        "_PLOT_CONDITION_INDEX"
    ],

    marker="o"
)


ax.axhline(

    CONDITION_INDEX_MODERATE_THRESHOLD,

    linestyle="--",

    linewidth=1,

    label=(
        f"Condition index = "
        f"{CONDITION_INDEX_MODERATE_THRESHOLD:.0f}"
    )
)


ax.axhline(

    CONDITION_INDEX_HIGH_THRESHOLD,

    linestyle="--",

    linewidth=1,

    label=(
        f"Condition index = "
        f"{CONDITION_INDEX_HIGH_THRESHOLD:.0f}"
    )
)


ax.set_xlabel(
    "Eigen-dimension"
)


ax.set_ylabel(
    "Condition index"
)


ax.set_title(
    "Condition indices - reduced multicollinearity diagnostic matrix"
)


ax.legend()


ax.grid(
    alpha=0.20
)


fig.tight_layout()


fig.savefig(

    CONDITION_INDEX_PATH,

    format="png",

    dpi=PNG_DPI,

    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 48. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 49. CONVERT IMAGES TO BASE64
# ============================================================

vif_base64 = (
    image_to_base64(
        VIF_PLOT_PATH
    )
)


full_eigenvalue_base64 = (
    image_to_base64(
        FULL_EIGENVALUE_PATH
    )
)


reduced_eigenvalue_base64 = (
    image_to_base64(
        REDUCED_EIGENVALUE_PATH
    )
)


condition_index_base64 = (
    image_to_base64(
        CONDITION_INDEX_PATH
    )
)


# ============================================================
# 50. HTML FLOAT FORMATTERS
# ============================================================

def format_float_or_infinity(
    value,
    decimals=6
):

    if pd.isna(
        value
    ):

        return (
            "NaN"
        )


    if np.isinf(
        value
    ):

        return (
            "Infinity"
        )


    return (
        f"{value:.{decimals}f}"
    )


# ============================================================
# 51. PREPARE HTML TABLES
# ============================================================

source_overview_html = (
    source_overview_table
    .to_html(
        index=False,
        border=0
    )
)


fewf_overview_html = (
    fewf_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "FALLBACK_VALUE":
                lambda value:
                    f"{value:.12f}"
        }
    )
)


fewf_fallback_usage_html = (
    fewf_fallback_usage_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "FALLBACK_VALUE":
                lambda value:
                    f"{value:.12f}"
        }
    )
)


feature_matrix_overview_html = (
    feature_matrix_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MEAN":
                lambda value:
                    f"{value:.8f}",

            "STANDARD_DEVIATION":
                lambda value:
                    f"{value:.8f}"
        }
    )
)


constant_features_html = (
    constant_features_table
    .to_html(
        index=False,
        border=0
    )
)


ohe_reference_html = (
    ohe_reference_table
    .to_html(
        index=False,
        border=0
    )
)


diagnostic_summary_html = (
    diagnostic_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "LARGEST_EIGENVALUE":
                lambda value:
                    format_float_or_infinity(
                        value,
                        10
                    ),

            "SMALLEST_EIGENVALUE":
                lambda value:
                    format_float_or_infinity(
                        value,
                        12
                    ),

            "CORRELATION_MATRIX_CONDITION_NUMBER":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    ),

            "STANDARDIZED_DESIGN_CONDITION_NUMBER":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    )
        }
    )
)


vif_html = (
    vif_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MULTIPLE_R_SQUARED":
                lambda value:
                    f"{value:.10f}",

            "TOLERANCE":
                lambda value:
                    f"{value:.10f}",

            "VIF":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    )
        }
    )
)


high_vif_html = (
    high_vif_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MULTIPLE_R_SQUARED":
                lambda value:
                    f"{value:.10f}",

            "TOLERANCE":
                lambda value:
                    f"{value:.10f}",

            "VIF":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    )
        }
    )
)


full_condition_html = (
    full_condition_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "EIGENVALUE":
                lambda value:
                    f"{value:.12f}",

            "CONDITION_INDEX":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    )
        }
    )
)


reduced_condition_html = (
    reduced_condition_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "EIGENVALUE":
                lambda value:
                    f"{value:.12f}",

            "CONDITION_INDEX":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    )
        }
    )
)


full_near_null_html = (
    full_near_null_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "EIGENVALUE":
                lambda value:
                    f"{value:.12f}",

            "CONDITION_INDEX":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    )
        }
    )
)


reduced_near_null_html = (
    reduced_near_null_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "EIGENVALUE":
                lambda value:
                    f"{value:.12f}",

            "CONDITION_INDEX":
                lambda value:
                    format_float_or_infinity(
                        value,
                        6
                    )
        }
    )
)


# ============================================================
# 52. MAIN SUMMARY VALUES
# ============================================================

full_rank_deficiency = (
    full_diagnostic_summary[
        "RANK_DEFICIENCY"
    ]
)


reduced_rank_deficiency = (
    reduced_diagnostic_summary[
        "RANK_DEFICIENCY"
    ]
)


reduced_design_condition_number = (
    reduced_diagnostic_summary[
        "STANDARDIZED_DESIGN_CONDITION_NUMBER"
    ]
)


number_high_vif = int(
    len(
        high_vif_table
    )
)


number_elevated_or_high_vif = int(
    len(
        elevated_or_high_vif_table
    )
)


# ============================================================
# 53. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Global Dataset Multicollinearity
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1600px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 50px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 12px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 7px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 50px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Global Dataset Structure —
Multicollinearity
</h1>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

<br>

<strong>Source features:</strong>
{len(REQUIRED_SOURCE_FEATURES)}

<br>

<strong>Full numerical matrix features:</strong>
{matrix_feature_count}

<br>

<strong>Reduced VIF diagnostic features:</strong>
{len(reduced_feature_names)}

</p>


<div class="note">

<strong>Excluded from all multicollinearity diagnostics:</strong>

<br><br>

{EXCLUDED_IDENTIFIER}

<br>

{EXCLUDED_TARGET}

<br><br>

NID_ALPHA is an identifier.

TARGET_OMEGA is the target and must remain outside the
unsupervised explanatory feature matrix.

</div>


<!-- ========================================================
     1. SOURCE MATRIX
========================================================= -->


<h2>
1. Source and encoded feature structure
</h2>


<div class="table-container">

{source_overview_html}

</div>


<div class="table-container">

{feature_matrix_overview_html}

</div>


<div class="note">

The parquet dataset is never modified.

Original numerical, geographic and cyclical features are
preserved.

Binary features are temporarily mapped to 0/1.

FEWF variables are generated temporarily from their
original categorical labels.

OHEWI variables are temporarily expanded into dummy
columns.

</div>


<!-- ========================================================
     2. FEWF
========================================================= -->


<h2>
2. Temporary FEWF structure
</h2>


<div class="table-container">

{fewf_overview_html}

</div>


<div class="table-container">

{fewf_fallback_usage_html}

</div>


<div class="note">

Frequency mappings are fitted on the exploratory dataset
only for this EDA.

In the final modeling pipeline, mappings for validation,
test or future observations should be fitted on training
data only.

</div>


<!-- ========================================================
     3. CONSTANT FEATURES
========================================================= -->


<h2>
3. Constant-feature diagnostic
</h2>


<div class="table-container">

{constant_features_html}

</div>


<div class="note">

A constant feature has zero variance and therefore cannot
contribute useful geometric information to PCA, t-SNE or
GMM.

Constant features are excluded from correlation-based
multicollinearity calculations.

No source feature is physically removed from the parquet
dataset during this analysis.

</div>


<!-- ========================================================
     4. FULL MATRIX STRUCTURAL DEPENDENCE
========================================================= -->


<h2>
4. Full encoded matrix structural dependence
</h2>


<div class="table-container">

{diagnostic_summary_html}

</div>


<p class="result">

Full matrix rank deficiency:
{full_rank_deficiency}

</p>


<div class="chart">

<img
    src="data:image/png;base64,{full_eigenvalue_base64}"
    alt="Full matrix eigenvalue spectrum"
>

</div>


<div class="table-container">

{full_condition_html}

</div>


<div class="note">

The full encoded matrix intentionally retains every
One-Hot dummy.

After centering, all dummies generated from the same source
categorical feature contain an exact structural linear
constraint.

For example, the centered weekday dummy columns jointly
sum to zero.

Therefore, rank deficiency in the complete One-Hot
representation is expected and should not automatically be
interpreted as an unexpected data-quality problem.

</div>


<!-- ========================================================
     5. OHE REFERENCE DUMMIES
========================================================= -->


<h2>
5. Temporary One-Hot reference-dummy adjustment
</h2>


<div class="table-container">

{ohe_reference_html}

</div>


<div class="note">

One dummy from each original One-Hot source is temporarily
excluded only for VIF and reduced conditioning diagnostics.

This removes the known dummy-variable structural
dependence so that remaining multicollinearity can be
studied more meaningfully.

<br><br>

This is a diagnostic transformation only.

It does not change dataset_final.parquet and does not yet
represent the final modeling decision.

</div>


<!-- ========================================================
     6. REDUCED MATRIX CONDITIONING
========================================================= -->


<h2>
6. Reduced diagnostic matrix conditioning
</h2>


<p class="result">

Reduced matrix rank deficiency:
{reduced_rank_deficiency}

<br>

Standardized design condition number:
{format_float_or_infinity(reduced_design_condition_number, 6)}

</p>


<div class="chart">

<img
    src="data:image/png;base64,{reduced_eigenvalue_base64}"
    alt="Reduced matrix eigenvalue spectrum"
>

</div>


<div class="chart">

<img
    src="data:image/png;base64,{condition_index_base64}"
    alt="Reduced matrix condition indices"
>

</div>


<div class="table-container">

{reduced_condition_html}

</div>


<div class="note">

Condition indices are calculated from eigenvalues of the
standardized correlation structure.

Exploratory reference guidelines used here:

<br><br>

<strong>
Condition index &lt; {CONDITION_INDEX_MODERATE_THRESHOLD:.0f}
</strong>
— low concern

<br>

<strong>
{CONDITION_INDEX_MODERATE_THRESHOLD:.0f} to
{CONDITION_INDEX_HIGH_THRESHOLD:.0f}
</strong>
— moderate concern

<br>

<strong>
&gt;= {CONDITION_INDEX_HIGH_THRESHOLD:.0f}
</strong>
— high concern

<br><br>

Infinite values indicate exact or numerically near-exact
linear dependence.

These thresholds are diagnostic guidelines rather than
automatic feature-removal rules.

</div>


<!-- ========================================================
     7. VIF
========================================================= -->


<h2>
7. Variance Inflation Factor
</h2>


<div class="table-container">

{vif_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{vif_base64}"
    alt="Variance Inflation Factor"
>

</div>


<p class="result">

Features with VIF >= {VIF_ELEVATED_THRESHOLD:.0f}:
{number_elevated_or_high_vif}

<br>

Features with VIF >= {VIF_HIGH_THRESHOLD:.0f}:
{number_high_vif}

</p>


<div class="note">

Variance Inflation Factor measures how well each feature can
be linearly explained by all remaining features.

<br><br>

The exploratory interpretation used here is:

<br><br>

<strong>
VIF &lt; {VIF_ELEVATED_THRESHOLD:.0f}
</strong>
— low

<br>

<strong>
{VIF_ELEVATED_THRESHOLD:.0f} to
{VIF_HIGH_THRESHOLD:.0f}
</strong>
— elevated

<br>

<strong>
VIF >= {VIF_HIGH_THRESHOLD:.0f}
</strong>
— high

<br><br>

A high VIF does not automatically require feature removal,
especially because the planned workflow includes PCA.

It identifies a feature whose information is strongly
recoverable from other variables.

</div>


<h3>
High-VIF features
</h3>


<div class="table-container">

{high_vif_html}

</div>


<!-- ========================================================
     8. NEAR-LINEAR DEPENDENCE
========================================================= -->


<h2>
8. Near-linear dependence components
</h2>


<h3>
Full encoded matrix
</h3>


<div class="table-container">

{full_near_null_html}

</div>


<h3>
Reduced diagnostic matrix
</h3>


<div class="table-container">

{reduced_near_null_html}

</div>


<div class="note">

Small eigenvalues identify directions in the feature space
with very little independent variance.

The listed features have the largest absolute coefficients
in the corresponding near-null eigenvectors.

<br><br>

This diagnostic can reveal multivariate linear dependence
that may not appear as a single pairwise correlation close
to 1.

</div>


<!-- ========================================================
     9. RELATION TO GLOBAL CORRELATION
========================================================= -->


<h2>
9. Relationship with the previous global-correlation analysis
</h2>


<div class="note">

The previous global-correlation stage evaluated
pairwise Pearson and Spearman relationships.

The current analysis asks a different question:

<br><br>

<strong>
Can one feature be explained by a combination of several
other features, and is the complete feature matrix close to
linear dependence?
</strong>

<br><br>

Therefore, VIF, eigenvalues, numerical rank and condition
indices complement rather than duplicate pairwise
correlation.

</div>


<!-- ========================================================
     10. CYCLICAL FEATURES
========================================================= -->


<h2>
10. Cyclical feature interpretation
</h2>


<div class="note">

TRANS_MONTH_SIN and TRANS_MONTH_COS jointly encode one
circular feature.

TRANS_HOUR_SIN and TRANS_HOUR_COS jointly encode another.

Their relationship is geometrically constrained by:

<br><br>

<strong>
sin² + cos² approximately equals 1
</strong>

<br><br>

This is a non-linear deterministic relationship and may
not produce a high VIF because VIF evaluates linear
predictability.

Therefore, a low VIF for sine and cosine does not imply
that the two components are unrelated.

They should continue to be interpreted as paired cyclical
representations.

</div>


<!-- ========================================================
     11. PCA
========================================================= -->


<h2>
11. Potential implications for PCA
</h2>


<div class="note">

PCA is specifically designed to reorganize correlated
variables into orthogonal principal components.

Therefore, multicollinearity is not an assumption violation
for PCA.

In fact, strong redundancy often causes variance to become
concentrated in fewer principal components.

<br><br>

However, exact or near-exact redundancy can create
extremely small eigenvalues and components carrying almost
no unique information.

The current diagnostics will therefore help determine
whether PCA naturally resolves the observed redundancy or
whether some representation should be adjusted before PCA.

</div>


<!-- ========================================================
     12. t-SNE
========================================================= -->


<h2>
12. Potential implications for t-SNE
</h2>


<div class="note">

t-SNE does not require absence of multicollinearity in the
classical regression sense.

However, correlated or repeated information can
effectively give some underlying dimensions greater weight
when distances and neighborhoods are calculated.

<br><br>

Therefore, redundancy detected here remains relevant to
the geometry supplied to t-SNE.

The final t-SNE representation should preferably be built
after the feature-space decisions and dimensionality
reduction strategy have been defined.

</div>


<!-- ========================================================
     13. GMM
========================================================= -->


<h2>
13. Potential implications for GMM
</h2>


<div class="note">

GMM is especially sensitive to redundant or nearly
dependent dimensions when covariance matrices are
estimated.

A nearly singular feature space may produce ill-conditioned
component covariance matrices.

This is especially important if full covariance is used.

<br><br>

The planned PCA-before-GMM strategy can substantially
reduce this problem by producing orthogonal components and
discarding directions with negligible variance.

The number of retained PCA components should therefore be
selected only after the pre-modeling audit.

</div>


<!-- ========================================================
     14. WHAT THIS ANALYSIS DOES NOT DO
========================================================= -->


<h2>
14. What this analysis does not do
</h2>


<div class="note">

No feature is removed from dataset_final.parquet.

No original category is overwritten.

No One-Hot dummy is permanently dropped.

No numerical transformation is permanently applied.

No target information is used.

No PCA, t-SNE or GMM model is fitted.

<br><br>

This stage is diagnostic only.

Its purpose is to identify structural redundancy and
potential numerical problems that must be reviewed before
the final modeling feature matrix is defined.

</div>


<!-- ========================================================
     15. SUMMARY
========================================================= -->


<h2>
15. Summary
</h2>


<p class="result">

Full encoded features:
{len(nonconstant_feature_names)}

</p>


<p class="result">

Reduced VIF diagnostic features:
{len(reduced_feature_names)}

</p>


<p class="result">

Temporarily excluded OHE reference dummies:
{len(ohe_reference_feature_names)}

</p>


<p class="result">

Full matrix rank deficiency:
{full_rank_deficiency}

</p>


<p class="result">

Reduced matrix rank deficiency:
{reduced_rank_deficiency}

</p>


<p class="result">

High-VIF features:
{number_high_vif}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The complete encoded feature structure was evaluated for
linear dependence using numerical rank, eigenvalues,
condition indices and Variance Inflation Factors.

<br><br>

The full One-Hot representation was analyzed first so that
its expected structural rank deficiency could be explicitly
documented.

One reference dummy per One-Hot source was then temporarily
excluded to create a reduced diagnostic matrix and separate
encoding-induced dependence from remaining
multicollinearity.

<br><br>

VIF evaluates whether an individual feature can be
linearly reconstructed from the remaining features.

Eigenvalue and condition-index diagnostics evaluate the
conditioning of the feature space as a whole.

Near-null eigenvectors identify combinations of variables
responsible for the weakest independent directions.

<br><br>

No feature-selection decision is made at this stage.

These findings should now be combined with the individual
EDA, within-group EDA, target-association EDA and global
correlation analysis during the final pre-modeling audit
before PCA, t-SNE and GMM.

</div>


</body>

</html>
"""


# ============================================================
# 54. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 55. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GLOBAL DATASET STRUCTURE - MULTICOLLINEARITY"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


print(
    "Full numerical matrix features:",
    matrix_feature_count
)


print(
    "Reduced VIF diagnostic features:",
    len(
        reduced_feature_names
    )
)


# ============================================================
# 56. DISPLAY OHE REFERENCE DUMMIES
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORARY OHE REFERENCE DUMMIES"
)


print(
    "=" * 100
)


display(
    ohe_reference_table
)


# ============================================================
# 57. DISPLAY DIAGNOSTIC SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MULTICOLLINEARITY DIAGNOSTIC SUMMARY"
)


print(
    "=" * 100
)


display(
    diagnostic_summary_table
)


# ============================================================
# 58. DISPLAY VIF
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "VARIANCE INFLATION FACTOR"
)


print(
    "=" * 100
)


display(
    vif_table
)


# ============================================================
# 59. DISPLAY HIGH VIF
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HIGH-VIF FEATURES"
)


print(
    "=" * 100
)


display(
    high_vif_table
)


# ============================================================
# 60. DISPLAY FULL CONDITION DIAGNOSTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "FULL MATRIX CONDITION DIAGNOSTICS"
)


print(
    "=" * 100
)


display(
    full_condition_table
)


# ============================================================
# 61. DISPLAY REDUCED CONDITION DIAGNOSTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "REDUCED MATRIX CONDITION DIAGNOSTICS"
)


print(
    "=" * 100
)


display(
    reduced_condition_table
)


# ============================================================
# 62. DISPLAY FULL NEAR-NULL COMPONENTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "FULL MATRIX NEAR-LINEAR COMPONENTS"
)


print(
    "=" * 100
)


display(
    full_near_null_table
)


# ============================================================
# 63. DISPLAY REDUCED NEAR-NULL COMPONENTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "REDUCED MATRIX NEAR-LINEAR COMPONENTS"
)


print(
    "=" * 100
)


display(
    reduced_near_null_table
)


# ============================================================
# 64. MAIN FINDINGS
# ============================================================

print(
    "\nFull matrix rank deficiency:",
    full_rank_deficiency
)


print(
    "Reduced matrix rank deficiency:",
    reduced_rank_deficiency
)


print(
    "High-VIF features:",
    number_high_vif
)


print(
    "Features with VIF >= 5:",
    number_elevated_or_high_vif
)


print(
    "Reduced standardized design condition number:",
    format_float_or_infinity(
        reduced_design_condition_number,
        6
    )
)


# ============================================================
# 65. RELEASE MEMORY
# ============================================================

del full_correlation_matrix
del full_nonconstant_correlation
del reduced_correlation_matrix

del full_eigenvalues
del full_eigenvectors

del reduced_eigenvalues
del reduced_eigenvectors

del global_matrix

gc.collect()


# ============================================================
# 66. REMOVE TEMPORARY DISK MATRIX
# ============================================================

if TEMPORARY_MATRIX_PATH.exists():

    TEMPORARY_MATRIX_PATH.unlink()


# ============================================================
# 67. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MULTICOLLINEARITY ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    VIF_PLOT_PATH
)


print(
    FULL_EIGENVALUE_PATH
)


print(
    REDUCED_EIGENVALUE_PATH
)


print(
    CONDITION_INDEX_PATH
)


print(
    "\nMachine-readable tables:"
)


print(
    VIF_CSV_PATH
)


print(
    HIGH_VIF_CSV_PATH
)


print(
    FULL_CONDITION_CSV_PATH
)


print(
    REDUCED_CONDITION_CSV_PATH
)


print(
    OHE_REFERENCE_CSV_PATH
)


print(
    FULL_NEAR_NULL_CSV_PATH
)


print(
    REDUCED_NEAR_NULL_CSV_PATH
)


print(
    MATRIX_OVERVIEW_CSV_PATH
)


print(
    "\nTemporary matrix removed:"
)


print(
    not TEMPORARY_MATRIX_PATH.exists()
)

Estimated temporary matrix size: 0.269 GiB

GLOBAL DATASET STRUCTURE - MULTICOLLINEARITY

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%
Full numerical matrix features: 39
Reduced VIF diagnostic features: 37

TEMPORARY OHE REFERENCE DUMMIES


,SOURCE_FEATURE,REFERENCE_CATEGORY,TEMPORARILY_EXCLUDED_DUMMY,PURPOSE
0,TRANS_WEEK_OHEWI,Friday,TRANS_WEEK_OHEWI_Friday,Remove structural OHE dependence for VIF diagn...
1,RECEIVE_CATEGORY_OHEWI,entertainment,RECEIVE_CATEGORY_OHEWI_entertainment,Remove structural OHE dependence for VIF diagn...



MULTICOLLINEARITY DIAGNOSTIC SUMMARY


,MATRIX,NUMBER_FEATURES,NUMERICAL_RANK,RANK_DEFICIENCY,LARGEST_EIGENVALUE,SMALLEST_EIGENVALUE,CORRELATION_MATRIX_CONDITION_NUMBER,STANDARDIZED_DESIGN_CONDITION_NUMBER
0,Full encoded matrix,39,37,2,2.510413,-1.703220e-16,inf,inf
1,Reduced VIF diagnostic matrix,37,37,0,2.510074,8.816608e-04,2846.983286,53.35713



VARIANCE INFLATION FACTOR


...


FULL MATRIX NEAR-LINEAR COMPONENTS


,SMALLEST_COMPONENT_RANK,EIGENVALUE,CONDITION_INDEX,TOP_CONTRIBUTING_FEATURES
0,1,-1.703220e-16,inf,TRANS_WEEK_OHEWI_Monday (0.328853); TRANS_WEEK...
1,2,1.802218e-16,inf,TRANS_WEEK_OHEWI_Monday (0.282868); TRANS_WEEK...
2,3,8.816608e-04,53.360741,SEND_LONG_REGISTER (0.707110); RECEIVE_LONG (-...
3,4,6.417449e-03,19.778412,SEND_LAT_REGISTER (0.707185); RECEIVE_LAT (-0....
4,5,3.066692e-02,9.047687,TRANS_NUM_CARD_FEWF (0.712661); SEND_NAME_FEWF...



REDUCED MATRIX NEAR-LINEAR COMPONENTS


,SMALLEST_COMPONENT_RANK,EIGENVALUE,CONDITION_INDEX,TOP_CONTRIBUTING_FEATURES
0,1,0.000882,53.357130,SEND_LONG_REGISTER (-0.707110); RECEIVE_LONG (...
1,2,0.006417,19.777073,SEND_LAT_REGISTER (-0.707185); RECEIVE_LAT (0....
2,3,0.030667,9.047082,TRANS_NUM_CARD_FEWF (-0.712662); SEND_NAME_FEW...
3,4,0.070097,5.984021,RECEIVE_CATEGORY_OHEWI_gas_transport (-0.41716...
4,5,0.127668,4.434074,RECEIVE_LOC_FEWF (0.684782); RECEIVE_CATEGORY_...



Full matrix rank deficiency: 2
Reduced matrix rank deficiency: 0
High-VIF features: 6
Features with VIF >= 5: 6
Reduced standardized design condition number: 53.357130

MULTICOLLINEARITY ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/multicollinearity

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/multicollinearity/analysis_multicollinearity.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/multicollinearity/multicollinearity_vif.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/multicollinearity/multicollinearity_full_eigenvalue_spectrum.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/03_global_dataset_structure/multicollinearity/multicollinearity_reduced_eigenvalue_spectrum.png
/projeto_tcc_2